# JT Segmenter

### Cell 1 — Imports, Environment Check, Data Loading

In [40]:
# ============================================================
# JT Segmenter — Mizo Sentence Segmentation
# Cell 1: Imports, environment, verified data pipeline, load all splits
# ============================================================

import os
import re
import sys
import json
import random
import warnings
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict
from difflib import SequenceMatcher

import numpy as np
import torch

warnings.filterwarnings("ignore")

# ---------- Reproducibility ----------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---------- Environment check ----------
print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)
print(f"Python            : {sys.version.split()[0]}")
print(f"PyTorch           : {torch.__version__}")
print(f"CUDA available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version      : {torch.version.cuda}")
    print(f"GPU               : {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"VRAM              : {vram_gb:.1f} GB")

try:
    import transformers
    print(f"Transformers      : {transformers.__version__}")
except ImportError:
    print("Transformers      : NOT INSTALLED")

try:
    import TorchCRF
    print(f"TorchCRF          : installed")
except ImportError:
    try:
        import torchcrf
        print(f"torchcrf          : installed")
    except ImportError:
        print("TorchCRF          : NOT INSTALLED")

try:
    import seqeval
    try:
        from importlib.metadata import version as _pkg_version
        print(f"Seqeval           : {_pkg_version('seqeval')}")
    except Exception:
        print(f"Seqeval           : installed (version unknown)")
except ImportError:
    print("Seqeval           : NOT INSTALLED")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device selected   : {DEVICE}")
print("=" * 60)

# ---------- Paths ----------
DATA_DIR = Path("data")
assert DATA_DIR.exists(), f"Data folder not found at {DATA_DIR.resolve()}"

# ---------- Quote/comma artefact set ----------
QUOTE_COMMA = {'"', ',', "'", '\u201c', '\u201d', '\u2018', '\u2019'}

# ============================================================
# Verified function 1: parse_paired_files
# ============================================================
def parse_paired_files(unseg_path, seg_path):
    """
    Parse unseg/seg file pair into aligned paragraph/sentence dicts.

    Returns: list[dict] with keys:
        - 'pid'       : paragraph index (int)
        - 'paragraph' : raw paragraph string (no <p>/</p> tags)
        - 'sentences' : list[str] of constituent sentences (no tags)
    """
    # --- Read unseg file: one paragraph per line wrapped in <p>...</p> ---
    with open(unseg_path, "r", encoding="utf-8") as f:
        unseg_lines = [ln.rstrip("\n") for ln in f if ln.strip()]

    paragraphs = []
    for ln in unseg_lines:
        s = ln.strip()
        if s.startswith("<p>"):
            s = s[3:]
        if s.endswith("</p>"):
            s = s[:-4]
        paragraphs.append(s.strip())

    # --- Read seg file: sentences split across lines, <p> on first sent of paragraph,
    #     </p> on last sent of paragraph ---
    with open(seg_path, "r", encoding="utf-8") as f:
        seg_lines = [ln.rstrip("\n") for ln in f if ln.strip()]

    grouped = []           # list of list[str] (sentences per paragraph)
    current = []
    in_para = False
    for ln in seg_lines:
        s = ln.strip()
        starts = s.startswith("<p>")
        ends = s.endswith("</p>")
        if starts:
            s = s[3:]
        if ends:
            s = s[:-4]
        s = s.strip()

        if starts:
            # new paragraph begins
            current = [s]
            in_para = True
            if ends:
                grouped.append(current)
                current = []
                in_para = False
        elif ends:
            current.append(s)
            grouped.append(current)
            current = []
            in_para = False
        else:
            if in_para:
                current.append(s)
            else:
                # defensive: stray sentence without <p>
                grouped.append([s])

    if len(paragraphs) != len(grouped):
        raise ValueError(
            f"Paragraph count mismatch: unseg={len(paragraphs)} vs seg={len(grouped)}\n"
            f"Files: {unseg_path} | {seg_path}"
        )

    aligned = []
    for i, (para, sents) in enumerate(zip(paragraphs, grouped)):
        aligned.append({
            "pid": i,
            "paragraph": para,
            "sentences": [s for s in sents if s],
        })
    return aligned


# ============================================================
# Verified function 2: label_paragraph_v5
# ============================================================
def _tokenize_ws(text):
    """Whitespace tokeniser preserving original tokens."""
    return text.split()


def _strip_quote_comma(tok):
    """Strip leading/trailing quote/comma artefacts; return cleaned token."""
    # Strip from both ends iteratively
    out = tok
    changed = True
    while changed and out:
        changed = False
        if out and out[0] in QUOTE_COMMA:
            out = out[1:]
            changed = True
        if out and out[-1] in QUOTE_COMMA:
            out = out[:-1]
            changed = True
    return out


def _fuzzy_tail_match(para_tokens, start_idx, sent_tokens, max_extra=3):
    """
    Try to align the tail of a sentence to paragraph tokens starting at start_idx.
    Returns end index (inclusive) in para_tokens of the final token of this sentence,
    or None if no acceptable match.
    """
    n_sent = len(sent_tokens)
    if n_sent == 0:
        return None

    # Try exact-length match first, then allow up to max_extra paragraph tokens
    # absorbed (e.g., trailing punctuation glued differently).
    best = None
    for extra in range(0, max_extra + 1):
        end_idx = start_idx + n_sent - 1 + extra
        if end_idx >= len(para_tokens):
            break
        # Compare last token (most important — that's the boundary token)
        para_last = _strip_quote_comma(para_tokens[end_idx])
        sent_last = _strip_quote_comma(sent_tokens[-1])
        if para_last == sent_last and para_last != "":
            # Confirm reasonable similarity over the span
            span = " ".join(para_tokens[start_idx:end_idx + 1])
            ref = " ".join(sent_tokens)
            ratio = SequenceMatcher(None, span, ref).ratio()
            if ratio >= 0.80:
                best = end_idx
                break
    return best


def label_paragraph_v5(paragraph, sentences):
    """
    Convert (paragraph, sentences) into (tokens, labels) where labels are
    binary: 1 = sentence-final token, 0 = non-boundary.

    Strategy:
      - Whitespace tokenise paragraph.
      - Walk through sentences in order; for each, find the index in paragraph
        tokens where its final token sits (handling quote/comma artefacts and
        small alignment drift via fuzzy_tail_match).
      - Mark that index with label 1.
    """
    para_tokens = _tokenize_ws(paragraph)
    n = len(para_tokens)
    labels = [0] * n

    cursor = 0
    for sent in sentences:
        sent_tokens = _tokenize_ws(sent)
        if not sent_tokens:
            continue

        end_idx = _fuzzy_tail_match(para_tokens, cursor, sent_tokens, max_extra=3)
        if end_idx is None:
            # Fallback: assume exact length
            end_idx = cursor + len(sent_tokens) - 1
            if end_idx >= n:
                end_idx = n - 1

        # Skip standalone quote/comma artefacts at end_idx — move boundary back
        # to the last "real" content token within this sentence's span.
        while end_idx > cursor and _strip_quote_comma(para_tokens[end_idx]) == "":
            end_idx -= 1

        labels[end_idx] = 1
        cursor = end_idx + 1

    # Ensure last token is labelled as boundary (paragraph always ends a sentence)
    if n > 0 and labels[-1] == 0:
        # Move boundary forward from last marked 1 to final non-artefact token
        last_real = n - 1
        while last_real > 0 and _strip_quote_comma(para_tokens[last_real]) == "":
            last_real -= 1
        # Clear any boundary past last_real and set last_real
        for i in range(last_real, n):
            labels[i] = 0
        labels[last_real] = 1

    return para_tokens, labels


# ============================================================
# Verified function 3: build_labeled_dataset_v5
# ============================================================
def build_labeled_dataset_v5(aligned, verbose=True, name=""):
    """
    Build full labelled dataset from aligned paragraph dicts.

    Returns: list[dict] with keys:
        - 'pid'    : paragraph id
        - 'tokens' : list[str]
        - 'labels' : list[int]   (0/1, same length as tokens)
        - 'n_sent' : int (number of sentences from gold)
    Plus prints a small QC summary.
    """
    dataset = []
    n_total_tokens = 0
    n_total_boundaries = 0
    n_failures = 0

    for item in aligned:
        try:
            tokens, labels = label_paragraph_v5(item["paragraph"], item["sentences"])
            if len(tokens) != len(labels):
                raise ValueError("token/label length mismatch")
            dataset.append({
                "pid": item["pid"],
                "tokens": tokens,
                "labels": labels,
                "n_sent": len(item["sentences"]),
            })
            n_total_tokens += len(tokens)
            n_total_boundaries += sum(labels)
        except Exception as e:
            n_failures += 1
            if verbose:
                print(f"  [FAIL] pid={item['pid']}: {e}")

    if verbose:
        ratio = (n_total_boundaries / n_total_tokens * 100) if n_total_tokens else 0.0
        tag = f"[{name}] " if name else ""
        print(f"{tag}paragraphs={len(dataset):,} | tokens={n_total_tokens:,} | "
              f"boundaries={n_total_boundaries:,} ({ratio:.2f}%) | failures={n_failures}")

    return dataset


# ============================================================
# Load all splits
# ============================================================
print("\n" + "=" * 60)
print("LOADING ALL DATA SPLITS")
print("=" * 60)

# --- Train ---
train_aligned = parse_paired_files(
    DATA_DIR / "training_unseg.txt",
    DATA_DIR / "training_seg.txt",
)
train_data = build_labeled_dataset_v5(train_aligned, name="train")

# --- Validation ---
val_aligned = parse_paired_files(
    DATA_DIR / "validation_unseg.txt",
    DATA_DIR / "validation_seg.txt",
)
val_data = build_labeled_dataset_v5(val_aligned, name="val")

# --- Test sets 1..5 ---
test_data = {}
for i in range(1, 6):
    aligned = parse_paired_files(
        DATA_DIR / f"test{i}_unseg.txt",
        DATA_DIR / f"test{i}_seg.txt",
    )
    test_data[f"test{i}"] = build_labeled_dataset_v5(aligned, name=f"test{i}")

# ============================================================
# Final summary
# ============================================================
print("\n" + "=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Train       : {len(train_data):,} paragraphs")
print(f"Validation  : {len(val_data):,} paragraphs")
for k, v in test_data.items():
    print(f"{k:<12}: {len(v):,} paragraphs")

# Quick sanity peek at one training example
print("\n" + "=" * 60)
print("SANITY PEEK — train_data[0]")
print("=" * 60)
ex = train_data[0]
print(f"pid     : {ex['pid']}")
print(f"n_sent  : {ex['n_sent']}")
print(f"n_tokens: {len(ex['tokens'])}")
print(f"first 20 tokens : {ex['tokens'][:20]}")
print(f"first 20 labels : {ex['labels'][:20]}")
print(f"boundary count  : {sum(ex['labels'])} (should equal n_sent={ex['n_sent']})")

print("\n✅ Cell 1 complete — ready for Cell 2 (tokenizer + label alignment for XLM-R).")

ENVIRONMENT CHECK
Python            : 3.10.19
PyTorch           : 2.7.1+cu118
CUDA available    : True
CUDA version      : 11.8
GPU               : NVIDIA GeForce RTX 3060
VRAM              : 12.0 GB
Transformers      : 4.57.6
torchcrf          : installed
Seqeval           : 1.2.2
Device selected   : cuda

LOADING ALL DATA SPLITS
[train] paragraphs=46,909 | tokens=3,069,899 | boundaries=181,348 (5.91%) | failures=0
[val] paragraphs=500 | tokens=35,732 | boundaries=2,087 (5.84%) | failures=0
[test1] paragraphs=100 | tokens=7,244 | boundaries=422 (5.83%) | failures=0
[test2] paragraphs=100 | tokens=6,072 | boundaries=363 (5.98%) | failures=0
[test3] paragraphs=100 | tokens=6,809 | boundaries=412 (6.05%) | failures=0
[test4] paragraphs=100 | tokens=7,810 | boundaries=438 (5.61%) | failures=0
[test5] paragraphs=100 | tokens=5,965 | boundaries=374 (6.27%) | failures=0

DATASET SUMMARY
Train       : 46,909 paragraphs
Validation  : 500 paragraphs
test1       : 100 paragraphs
test2       : 10

### Cell 2 — Tokenizer & Subword Label Alignment

In [41]:
# ============================================================
# Cell 2: XLM-R tokenizer + subword-to-word label alignment
# ============================================================

from transformers import AutoTokenizer

MODEL_NAME  = "xlm-roberta-base"
MAX_LEN     = 256          # subword tokens per chunk
STRIDE      = 64           # overlap between chunks for long paragraphs
LABEL_PAD   = -100         # ignored by both CRF mask and CrossEntropy

print("=" * 60)
print(f"Loading tokenizer: {MODEL_NAME}")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print(f"Tokenizer class       : {tokenizer.__class__.__name__}")
print(f"Vocab size            : {tokenizer.vocab_size:,}")
print(f"Model max length      : {tokenizer.model_max_length}")
print(f"Pad / CLS / SEP / UNK : "
      f"{tokenizer.pad_token_id} / {tokenizer.cls_token_id} / "
      f"{tokenizer.sep_token_id} / {tokenizer.unk_token_id}")


# ============================================================
# Subword-to-word alignment with chunking
# ============================================================
def encode_paragraph(tokens, labels, tokenizer,
                     max_len=MAX_LEN, stride=STRIDE, label_pad=LABEL_PAD):
    """
    Encode a single paragraph (pre-tokenised words + word-level labels) into
    one or more subword chunks suitable for XLM-R.

    For each subword:
        - First subword of a word inherits that word's label (0 or 1).
        - Continuation subwords get label_pad (-100), so they are ignored
          during loss computation and CRF emission masking.
        - Special tokens ([CLS], [SEP], padding) get label_pad.

    Long paragraphs are split into overlapping chunks of `max_len` subwords
    with `stride` overlap. Boundary labels in overlap regions are present in
    both chunks; this is fine for training (CRF sees them) and we will
    de-duplicate at inference time using offset tracking.

    Returns: list[dict] — each dict is one chunk with:
        - input_ids       : list[int]   length <= max_len
        - attention_mask  : list[int]   length <= max_len
        - labels          : list[int]   length <= max_len, -100 padded
        - word_ids        : list[int|None]  per-subword word index (or None)
        - chunk_idx       : int (0-based within this paragraph)
        - n_chunks        : int (filled in after all chunks built)
    """
    enc = tokenizer(
        tokens,
        is_split_into_words=True,
        return_offsets_mapping=False,
        truncation=False,
        add_special_tokens=False,   # we'll wrap chunks ourselves
    )
    sub_ids   = enc["input_ids"]
    word_ids  = enc.word_ids()      # one entry per subword, None for specials

    # Build per-subword labels (first-subword-of-word strategy)
    sub_labels = []
    prev_w = None
    for w in word_ids:
        if w is None:
            sub_labels.append(label_pad)
        elif w != prev_w:
            sub_labels.append(labels[w])     # first subword inherits word label
            prev_w = w
        else:
            sub_labels.append(label_pad)     # continuation
    assert len(sub_ids) == len(sub_labels) == len(word_ids)

    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id
    inner_max = max_len - 2          # leave room for [CLS] and [SEP]

    chunks = []
    if len(sub_ids) <= inner_max:
        ranges = [(0, len(sub_ids))]
    else:
        ranges = []
        start = 0
        while start < len(sub_ids):
            end = min(start + inner_max, len(sub_ids))
            ranges.append((start, end))
            if end == len(sub_ids):
                break
            start = end - stride     # overlap

    for ci, (s, e) in enumerate(ranges):
        ids   = [cls_id] + sub_ids[s:e]   + [sep_id]
        labs  = [label_pad] + sub_labels[s:e] + [label_pad]
        wids  = [None] + word_ids[s:e]    + [None]
        attn  = [1] * len(ids)

        chunks.append({
            "input_ids":      ids,
            "attention_mask": attn,
            "labels":         labs,
            "word_ids":       wids,
            "chunk_idx":      ci,
            "n_chunks":       len(ranges),
        })
    return chunks


def build_encoded_dataset(labeled_data, tokenizer, name="",
                          max_len=MAX_LEN, stride=STRIDE):
    """
    Encode an entire split. Returns a flat list of chunks; each chunk also
    carries 'pid' and 'n_words' so we can reassemble paragraph-level
    predictions later.
    """
    flat = []
    n_long = 0
    n_chunks_total = 0
    for item in labeled_data:
        chunks = encode_paragraph(item["tokens"], item["labels"], tokenizer,
                                  max_len=max_len, stride=stride)
        if len(chunks) > 1:
            n_long += 1
        for ch in chunks:
            ch["pid"]     = item["pid"]
            ch["n_words"] = len(item["tokens"])
            flat.append(ch)
        n_chunks_total += len(chunks)

    if name:
        print(f"[{name}] paragraphs={len(labeled_data):,} | "
              f"chunks={n_chunks_total:,} | "
              f"long-paragraphs(>{max_len} subwords)={n_long:,}")
    return flat


# ============================================================
# Encode all splits
# ============================================================
print("\n" + "=" * 60)
print("ENCODING ALL SPLITS")
print("=" * 60)

train_enc = build_encoded_dataset(train_data, tokenizer, name="train")
val_enc   = build_encoded_dataset(val_data,   tokenizer, name="val")
test_enc  = {k: build_encoded_dataset(v, tokenizer, name=k)
             for k, v in test_data.items()}


# ============================================================
# Sanity check — round-trip alignment on train_enc[0]
# ============================================================
print("\n" + "=" * 60)
print("SANITY CHECK — subword alignment on train_data[0]")
print("=" * 60)

ex_words  = train_data[0]["tokens"]
ex_labels = train_data[0]["labels"]
chunks0   = [c for c in train_enc if c["pid"] == 0]
print(f"Paragraph 0: {len(ex_words)} words → {len(chunks0)} chunk(s)")

# Reconstruct word-level labels from first-subword positions
recovered = [None] * len(ex_words)
for ch in chunks0:
    for sub_lab, w in zip(ch["labels"], ch["word_ids"]):
        if w is not None and sub_lab != LABEL_PAD and recovered[w] is None:
            recovered[w] = sub_lab

mismatches = [i for i, (a, b) in enumerate(zip(ex_labels, recovered)) if a != b]
n_recovered = sum(1 for x in recovered if x is not None)
print(f"Words covered by first-subword labels : {n_recovered}/{len(ex_words)}")
print(f"Mismatches vs gold word labels         : {len(mismatches)}")
assert n_recovered == len(ex_words), "Some words missing first-subword label!"
assert not mismatches, f"Label mismatches at word indices: {mismatches[:10]}"

# Show subword view of first 10 words
first_chunk = chunks0[0]
print("\nFirst 25 subwords of chunk 0:")
print(f"{'idx':>3}  {'subword':<20}  {'word_id':>7}  {'label':>5}")
for i in range(min(25, len(first_chunk["input_ids"]))):
    sw = tokenizer.convert_ids_to_tokens(first_chunk["input_ids"][i])
    wid = first_chunk["word_ids"][i]
    lab = first_chunk["labels"][i]
    print(f"{i:>3}  {sw:<20}  {str(wid):>7}  {str(lab):>5}")

# Length distribution
chunk_lens = [len(c["input_ids"]) for c in train_enc]
print(f"\nChunk length stats (train): "
      f"min={min(chunk_lens)} max={max(chunk_lens)} "
      f"mean={np.mean(chunk_lens):.1f} median={int(np.median(chunk_lens))}")

print("\n✅ Cell 2 complete — encoded splits ready for Cell 3 (PyTorch Dataset + DataLoader).")

Loading tokenizer: xlm-roberta-base


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (621 > 512). Running this sequence through the model will result in indexing errors


Tokenizer class       : XLMRobertaTokenizerFast
Vocab size            : 250,002
Model max length      : 512
Pad / CLS / SEP / UNK : 1 / 0 / 2 / 3

ENCODING ALL SPLITS
[train] paragraphs=46,909 | chunks=57,218 | long-paragraphs(>256 subwords)=6,829
[val] paragraphs=500 | chunks=634 | long-paragraphs(>256 subwords)=86
[test1] paragraphs=100 | chunks=131 | long-paragraphs(>256 subwords)=17
[test2] paragraphs=100 | chunks=117 | long-paragraphs(>256 subwords)=15
[test3] paragraphs=100 | chunks=124 | long-paragraphs(>256 subwords)=13
[test4] paragraphs=100 | chunks=131 | long-paragraphs(>256 subwords)=16
[test5] paragraphs=100 | chunks=115 | long-paragraphs(>256 subwords)=12

SANITY CHECK — subword alignment on train_data[0]
Paragraph 0: 168 words → 2 chunk(s)
Words covered by first-subword labels : 168/168
Mismatches vs gold word labels         : 0

First 25 subwords of chunk 0:
idx  subword               word_id  label
  0  <s>                      None   -100
  1  ▁He                     

### Cell 3 — PyTorch Dataset, Collator, DataLoaders

In [43]:
# ============================================================
# Cell 3: PyTorch Dataset, dynamic-padding collator, DataLoaders
# ============================================================

from torch.utils.data import Dataset, DataLoader

# ---------- Hyperparameters for batching ----------
BATCH_SIZE_TRAIN = 16      # 256 subwords × 16 ≈ comfortable on RTX 3060 12GB
BATCH_SIZE_EVAL  = 32      # eval has no grads, can afford larger
NUM_WORKERS      = 0       # Windows + Jupyter: keep 0 to avoid spawn issues
PIN_MEMORY       = True    # speeds host→GPU transfer when CUDA is available

PAD_ID = tokenizer.pad_token_id   # XLM-R pad id = 1


# ============================================================
# Dataset
# ============================================================
class MizoSegDataset(Dataset):
    """
    Wraps the flat list of encoded chunks produced by build_encoded_dataset().
    Each item returns raw Python lists; the collator handles tensorisation
    and dynamic padding.
    """
    def __init__(self, encoded_chunks):
        self.data = encoded_chunks

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ch = self.data[idx]
        return {
            "input_ids":      ch["input_ids"],
            "attention_mask": ch["attention_mask"],
            "labels":         ch["labels"],
            # metadata kept for inference-time reassembly:
            "word_ids":       ch["word_ids"],
            "pid":            ch["pid"],
            "chunk_idx":      ch["chunk_idx"],
            "n_chunks":       ch["n_chunks"],
            "n_words":        ch["n_words"],
        }


# ============================================================
# Collator — dynamic padding to longest in batch
# ============================================================
def collate_fn(batch):
    """
    Pad input_ids with PAD_ID, attention_mask with 0, labels with LABEL_PAD (-100).
    Returns a dict of tensors for the model + a 'meta' list for reassembly.
    """
    max_len = max(len(item["input_ids"]) for item in batch)

    input_ids_b      = []
    attention_mask_b = []
    labels_b         = []
    meta             = []

    for item in batch:
        L   = len(item["input_ids"])
        pad = max_len - L

        input_ids_b.append(     item["input_ids"]      + [PAD_ID]    * pad)
        attention_mask_b.append(item["attention_mask"] + [0]         * pad)
        labels_b.append(        item["labels"]         + [LABEL_PAD] * pad)

        meta.append({
            "word_ids":  item["word_ids"] + [None] * pad,
            "pid":       item["pid"],
            "chunk_idx": item["chunk_idx"],
            "n_chunks":  item["n_chunks"],
            "n_words":   item["n_words"],
            "valid_len": L,
        })

    return {
        "input_ids":      torch.tensor(input_ids_b,      dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask_b, dtype=torch.long),
        "labels":         torch.tensor(labels_b,         dtype=torch.long),
        "meta":           meta,
    }


# ============================================================
# Build datasets & dataloaders
# ============================================================
print("=" * 60)
print("BUILDING DATASETS & DATALOADERS")
print("=" * 60)

train_ds = MizoSegDataset(train_enc)
val_ds   = MizoSegDataset(val_enc)
test_ds  = {k: MizoSegDataset(v) for k, v in test_enc.items()}

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE_TRAIN,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn,
    drop_last=False,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE_EVAL,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn,
)

test_loaders = {
    k: DataLoader(
        ds,
        batch_size=BATCH_SIZE_EVAL,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_fn,
    )
    for k, ds in test_ds.items()
}

print(f"train_loader : {len(train_ds):,} chunks → {len(train_loader):,} batches "
      f"(batch_size={BATCH_SIZE_TRAIN})")
print(f"val_loader   : {len(val_ds):,} chunks → {len(val_loader):,} batches "
      f"(batch_size={BATCH_SIZE_EVAL})")
for k, ds in test_ds.items():
    print(f"{k:<12}: {len(ds):,} chunks → {len(test_loaders[k]):,} batches "
          f"(batch_size={BATCH_SIZE_EVAL})")


# ============================================================
# Sanity check — pull one training batch
# ============================================================
print("\n" + "=" * 60)
print("SANITY CHECK — one training batch")
print("=" * 60)

# Use a deterministic loader for the peek (don't disturb the shuffled one)
peek_loader = DataLoader(train_ds, batch_size=BATCH_SIZE_TRAIN,
                         shuffle=False, collate_fn=collate_fn)
batch = next(iter(peek_loader))

print(f"input_ids       : {tuple(batch['input_ids'].shape)}  "
      f"dtype={batch['input_ids'].dtype}")
print(f"attention_mask  : {tuple(batch['attention_mask'].shape)}  "
      f"dtype={batch['attention_mask'].dtype}")
print(f"labels          : {tuple(batch['labels'].shape)}  "
      f"dtype={batch['labels'].dtype}")
print(f"meta entries    : {len(batch['meta'])}")

# Per-position label distribution within this batch
flat_labs = batch["labels"].view(-1).tolist()
n_neg  = sum(1 for x in flat_labs if x == 0)
n_pos  = sum(1 for x in flat_labs if x == 1)
n_pad  = sum(1 for x in flat_labs if x == LABEL_PAD)
print(f"label counts    : 0={n_neg:,}  1={n_pos:,}  -100={n_pad:,}")
if n_pos > 0:
    print(f"pos:neg ratio   : 1:{n_neg / n_pos:.1f}  "
          f"(expected ~1:16 over the corpus)")

# Verify no out-of-range token ids
assert batch["input_ids"].min() >= 0
assert batch["input_ids"].max() < tokenizer.vocab_size
assert set(batch["attention_mask"].unique().tolist()) <= {0, 1}

# Verify -100 only appears where attention_mask is 0 OR on continuation/special positions
# (i.e., never on a "real" first-subword position)
assert ((batch["labels"] == LABEL_PAD) |
        ((batch["labels"] == 0) | (batch["labels"] == 1))).all().item()

# Quick GPU transfer test
batch_gpu = {k: v.to(DEVICE) for k, v in batch.items() if isinstance(v, torch.Tensor)}
print(f"GPU transfer OK : input_ids on {batch_gpu['input_ids'].device}")

# Cleanup the peek tensors so we don't hold VRAM
del batch_gpu
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n✅ Cell 3 complete — DataLoaders ready for Cell 4 (model definition).")

BUILDING DATASETS & DATALOADERS
train_loader : 57,218 chunks → 3,577 batches (batch_size=16)
val_loader   : 634 chunks → 20 batches (batch_size=32)
test1       : 131 chunks → 5 batches (batch_size=32)
test2       : 117 chunks → 4 batches (batch_size=32)
test3       : 124 chunks → 4 batches (batch_size=32)
test4       : 131 chunks → 5 batches (batch_size=32)
test5       : 115 chunks → 4 batches (batch_size=32)

SANITY CHECK — one training batch
input_ids       : (16, 256)  dtype=torch.int64
attention_mask  : (16, 256)  dtype=torch.int64
labels          : (16, 256)  dtype=torch.int64
meta entries    : 16
label counts    : 0=1,069  1=62  -100=2,965
pos:neg ratio   : 1:17.2  (expected ~1:16 over the corpus)
GPU transfer OK : input_ids on cuda:0

✅ Cell 3 complete — DataLoaders ready for Cell 4 (model definition).


### Cell 4 — JT Segmenter Model (XLM-R + CRF)

In [44]:
# ============================================================
# Cell 4: JT Segmenter — XLM-R-base + linear head + CRF
# ============================================================

import torch.nn as nn
from transformers import AutoModel, AutoConfig

# torchcrf API: CRF(num_tags, batch_first=True)
# - forward(emissions, tags, mask, reduction='mean') → log-likelihood (scalar)
# - decode(emissions, mask) → list[list[int]] of best tag sequences
from torchcrf import CRF

NUM_LABELS = 2          # 0 = non-boundary, 1 = sentence-final
DROPOUT    = 0.1


class JTSegmenter(nn.Module):
    """
    JT Segmenter — Mizo sentence segmentation model.

    Architecture:
        XLM-R-base  →  dropout  →  linear(hidden → 2)  →  CRF

    Forward modes:
        - training (labels provided)  → returns dict with 'loss' = NLL
        - inference (labels=None)     → returns dict with 'predictions'
                                        = list[list[int]] of decoded tags
                                        (one list per sequence, of variable
                                         length equal to that sequence's
                                         valid scoring positions)

    CRF mask handling:
        Positions with label == -100 are NOT scored by the CRF. We build a
        boolean `crf_mask` that is True only at "real" first-subword positions
        (i.e., where labels != -100). This prevents the CRF from learning
        spurious transitions through padding / continuation subwords.

        torchcrf requires that `crf_mask[:, 0]` be True for every sequence.
        That is guaranteed in our setup because:
          - position 0 is always <s> with label -100  → would be False
        ...so we cannot simply pass a mask that starts at <s>. Instead we
        gather only the valid (non -100) positions for each sequence into a
        compact, left-aligned tensor before feeding the CRF. This is the
        standard trick for token-classification + CRF with HuggingFace-style
        ignore-index labelling.
    """

    def __init__(self, model_name=MODEL_NAME, num_labels=NUM_LABELS,
                 dropout=DROPOUT):
        super().__init__()
        self.config   = AutoConfig.from_pretrained(model_name)
        self.encoder  = AutoModel.from_pretrained(model_name)
        self.dropout  = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)
        self.crf      = CRF(num_labels, batch_first=True)
        self.num_labels = num_labels

    # ---------- helper: pack (-100 aware) sequences for the CRF ----------
    @staticmethod
    def _pack_for_crf(emissions, labels):
        """
        Gather valid (label != -100) positions from each sequence into a
        left-aligned, padded tensor suitable for torchcrf.

        Args:
            emissions : (B, T, C) float
            labels    : (B, T)    long, with -100 at ignored positions
        Returns:
            packed_emissions : (B, T', C)
            packed_labels    : (B, T')   long, 0 at padded positions
            packed_mask      : (B, T')   bool, True at real positions
        """
        B, T, C = emissions.shape
        valid   = labels.ne(-100)                  # (B, T) bool
        lengths = valid.sum(dim=1)                 # (B,)
        max_len = int(lengths.max().item())

        device = emissions.device
        packed_em   = torch.zeros(B, max_len, C, device=device,
                                  dtype=emissions.dtype)
        packed_lab  = torch.zeros(B, max_len, device=device, dtype=torch.long)
        packed_mask = torch.zeros(B, max_len, device=device, dtype=torch.bool)

        for i in range(B):
            n = int(lengths[i].item())
            if n == 0:
                # No valid positions in this sequence — should not happen, but
                # we guard against it. Mark first slot True with label 0 to
                # satisfy CRF's "mask[:,0] must be True" invariant; this row
                # will contribute negligible loss and we can filter later.
                packed_mask[i, 0] = True
                continue
            sel = valid[i].nonzero(as_tuple=False).squeeze(1)   # (n,)
            packed_em[i, :n]   = emissions[i, sel]
            packed_lab[i, :n]  = labels[i, sel]
            packed_mask[i, :n] = True

        return packed_em, packed_lab, packed_mask, lengths

    # ---------- forward ----------
    def forward(self, input_ids, attention_mask, labels=None):
        out = self.encoder(input_ids=input_ids,
                           attention_mask=attention_mask)
        hidden    = self.dropout(out.last_hidden_state)        # (B, T, H)
        emissions = self.classifier(hidden)                    # (B, T, C)

        if labels is not None:
            packed_em, packed_lab, packed_mask, lengths = \
                self._pack_for_crf(emissions, labels)

            # Negative log-likelihood (mean over batch, summed over time
            # within each sequence by torchcrf with reduction='mean').
            log_lik = self.crf(packed_em, packed_lab,
                               mask=packed_mask, reduction="mean")
            loss = -log_lik
            return {"loss": loss, "emissions": emissions}

        # Inference: pack with a "fake" labels tensor (all valid where labels
        # would be valid) — but at inference we don't have labels. We instead
        # use attention_mask to identify real (non-pad) positions, and rely on
        # the caller to extract first-subword predictions using word_ids.
        # However, the CRF will still emit a prediction at EVERY masked position
        # including continuation subwords. That's fine: downstream code keeps
        # only the first-subword prediction per word.
        crf_mask = attention_mask.bool()
        # CRF requires mask[:, 0] == True; <s> always has attention 1, so OK.
        decoded = self.crf.decode(emissions, mask=crf_mask)
        return {"predictions": decoded, "emissions": emissions}


# ============================================================
# Build model
# ============================================================
print("=" * 60)
print("BUILDING JT SEGMENTER")
print("=" * 60)

model = JTSegmenter(model_name=MODEL_NAME,
                    num_labels=NUM_LABELS,
                    dropout=DROPOUT).to(DEVICE)

# Parameter counts
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
encoder_params   = sum(p.numel() for p in model.encoder.parameters())
head_params      = (sum(p.numel() for p in model.classifier.parameters())
                    + sum(p.numel() for p in model.crf.parameters()))

print(f"Encoder    : {MODEL_NAME}")
print(f"Hidden size: {model.config.hidden_size}")
print(f"Num labels : {NUM_LABELS}")
print(f"Dropout    : {DROPOUT}")
print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable_params:,}")
print(f"  └─ Encoder (XLM-R-base) : {encoder_params:,}")
print(f"  └─ Linear + CRF head    : {head_params:,}")

# CRF transition / start / end matrices (sanity peek)
print("\nCRF parameters (initial, before training):")
print(f"  start_transitions : {model.crf.start_transitions.detach().cpu().numpy()}")
print(f"  end_transitions   : {model.crf.end_transitions.detach().cpu().numpy()}")
print(f"  transitions (2x2) :\n{model.crf.transitions.detach().cpu().numpy()}")


# ============================================================
# Forward-pass smoke tests
# ============================================================
print("\n" + "=" * 60)
print("SMOKE TEST — training forward")
print("=" * 60)

model.train()
batch = next(iter(peek_loader))
batch_gpu = {k: v.to(DEVICE) for k, v in batch.items() if isinstance(v, torch.Tensor)}

with torch.no_grad():
    out = model(input_ids=batch_gpu["input_ids"],
                attention_mask=batch_gpu["attention_mask"],
                labels=batch_gpu["labels"])

print(f"loss        : {out['loss'].item():.4f}")
print(f"emissions   : {tuple(out['emissions'].shape)}")
# Expected initial loss for 2-class CRF over ~1100 valid tokens: ~0.69 * mean_len
# (model is roughly chance until trained). A finite, positive value is enough.
assert torch.isfinite(out["loss"]), "Loss is NaN/Inf!"

print("\n" + "=" * 60)
print("SMOKE TEST — inference forward")
print("=" * 60)

model.eval()
with torch.no_grad():
    out = model(input_ids=batch_gpu["input_ids"],
                attention_mask=batch_gpu["attention_mask"],
                labels=None)

preds = out["predictions"]
print(f"# decoded sequences : {len(preds)}")
print(f"first sequence len  : {len(preds[0])}  "
      f"(matches attention_mask sum: {int(batch_gpu['attention_mask'][0].sum().item())})")
print(f"first 30 predicted tags : {preds[0][:30]}")

# At init, CRF will tend to predict all-0 (majority class). That's fine.
n_pos_pred = sum(sum(1 for t in seq if t == 1) for seq in preds)
n_total    = sum(len(seq) for seq in preds)
print(f"predicted positives : {n_pos_pred} / {n_total}  "
      f"(expect mostly 0s before training)")

# VRAM check
if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)
    print(f"\nGPU memory: allocated={alloc:.2f} GB | reserved={reserved:.2f} GB")

# Cleanup
del out, batch_gpu
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n✅ Cell 4 complete — JT Segmenter ready for Cell 5 (training loop).")

BUILDING JT SEGMENTER


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Encoder    : xlm-roberta-base
Hidden size: 768
Num labels : 2
Dropout    : 0.1
Total params     : 278,045,194
Trainable params : 278,045,194
  └─ Encoder (XLM-R-base) : 278,043,648
  └─ Linear + CRF head    : 1,546

CRF parameters (initial, before training):
  start_transitions : [-0.03565486  0.095664  ]
  end_transitions   : [0.03835721 0.09841133]
  transitions (2x2) :
[[0.02741956 0.02264313]
 [0.04110936 0.01716325]]

SMOKE TEST — training forward
loss        : 53.1510
emissions   : (16, 256, 2)

SMOKE TEST — inference forward
# decoded sequences : 16
first sequence len  : 256  (matches attention_mask sum: 256)
first 30 predicted tags : [1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1]
predicted positives : 1448 / 2008  (expect mostly 0s before training)

GPU memory: allocated=1.04 GB | reserved=1.30 GB

✅ Cell 4 complete — JT Segmenter ready for Cell 5 (training loop).


### Cell 4: JT Segmenter — XLM-R-base + linear head + CRF

In [45]:
# ============================================================
# Cell 4: JT Segmenter — XLM-R-base + linear head + CRF
# ============================================================

import torch.nn as nn
from transformers import AutoModel, AutoConfig

# torchcrf API: CRF(num_tags, batch_first=True)
# - forward(emissions, tags, mask, reduction='mean') → log-likelihood (scalar)
# - decode(emissions, mask) → list[list[int]] of best tag sequences
from torchcrf import CRF

NUM_LABELS = 2          # 0 = non-boundary, 1 = sentence-final
DROPOUT    = 0.1


class JTSegmenter(nn.Module):
    """
    JT Segmenter — Mizo sentence segmentation model.

    Architecture:
        XLM-R-base  →  dropout  →  linear(hidden → 2)  →  CRF

    Forward modes:
        - training (labels provided)  → returns dict with 'loss' = NLL
        - inference (labels=None)     → returns dict with 'predictions'
                                        = list[list[int]] of decoded tags
                                        (one list per sequence, of variable
                                         length equal to that sequence's
                                         valid scoring positions)

    CRF mask handling:
        Positions with label == -100 are NOT scored by the CRF. We build a
        boolean `crf_mask` that is True only at "real" first-subword positions
        (i.e., where labels != -100). This prevents the CRF from learning
        spurious transitions through padding / continuation subwords.

        torchcrf requires that `crf_mask[:, 0]` be True for every sequence.
        That is guaranteed in our setup because:
          - position 0 is always <s> with label -100  → would be False
        ...so we cannot simply pass a mask that starts at <s>. Instead we
        gather only the valid (non -100) positions for each sequence into a
        compact, left-aligned tensor before feeding the CRF. This is the
        standard trick for token-classification + CRF with HuggingFace-style
        ignore-index labelling.
    """

    def __init__(self, model_name=MODEL_NAME, num_labels=NUM_LABELS,
                 dropout=DROPOUT):
        super().__init__()
        self.config   = AutoConfig.from_pretrained(model_name)
        self.encoder  = AutoModel.from_pretrained(model_name)
        self.dropout  = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)
        self.crf      = CRF(num_labels, batch_first=True)
        self.num_labels = num_labels

    # ---------- helper: pack (-100 aware) sequences for the CRF ----------
    @staticmethod
    def _pack_for_crf(emissions, labels):
        """
        Gather valid (label != -100) positions from each sequence into a
        left-aligned, padded tensor suitable for torchcrf.

        Args:
            emissions : (B, T, C) float
            labels    : (B, T)    long, with -100 at ignored positions
        Returns:
            packed_emissions : (B, T', C)
            packed_labels    : (B, T')   long, 0 at padded positions
            packed_mask      : (B, T')   bool, True at real positions
        """
        B, T, C = emissions.shape
        valid   = labels.ne(-100)                  # (B, T) bool
        lengths = valid.sum(dim=1)                 # (B,)
        max_len = int(lengths.max().item())

        device = emissions.device
        packed_em   = torch.zeros(B, max_len, C, device=device,
                                  dtype=emissions.dtype)
        packed_lab  = torch.zeros(B, max_len, device=device, dtype=torch.long)
        packed_mask = torch.zeros(B, max_len, device=device, dtype=torch.bool)

        for i in range(B):
            n = int(lengths[i].item())
            if n == 0:
                # No valid positions in this sequence — should not happen, but
                # we guard against it. Mark first slot True with label 0 to
                # satisfy CRF's "mask[:,0] must be True" invariant; this row
                # will contribute negligible loss and we can filter later.
                packed_mask[i, 0] = True
                continue
            sel = valid[i].nonzero(as_tuple=False).squeeze(1)   # (n,)
            packed_em[i, :n]   = emissions[i, sel]
            packed_lab[i, :n]  = labels[i, sel]
            packed_mask[i, :n] = True

        return packed_em, packed_lab, packed_mask, lengths

    # ---------- forward ----------
    def forward(self, input_ids, attention_mask, labels=None):
        out = self.encoder(input_ids=input_ids,
                           attention_mask=attention_mask)
        hidden    = self.dropout(out.last_hidden_state)        # (B, T, H)
        emissions = self.classifier(hidden)                    # (B, T, C)

        if labels is not None:
            packed_em, packed_lab, packed_mask, lengths = \
                self._pack_for_crf(emissions, labels)

            # Negative log-likelihood (mean over batch, summed over time
            # within each sequence by torchcrf with reduction='mean').
            log_lik = self.crf(packed_em, packed_lab,
                               mask=packed_mask, reduction="mean")
            loss = -log_lik
            return {"loss": loss, "emissions": emissions}

        # Inference: pack with a "fake" labels tensor (all valid where labels
        # would be valid) — but at inference we don't have labels. We instead
        # use attention_mask to identify real (non-pad) positions, and rely on
        # the caller to extract first-subword predictions using word_ids.
        # However, the CRF will still emit a prediction at EVERY masked position
        # including continuation subwords. That's fine: downstream code keeps
        # only the first-subword prediction per word.
        crf_mask = attention_mask.bool()
        # CRF requires mask[:, 0] == True; <s> always has attention 1, so OK.
        decoded = self.crf.decode(emissions, mask=crf_mask)
        return {"predictions": decoded, "emissions": emissions}


# ============================================================
# Build model
# ============================================================
print("=" * 60)
print("BUILDING JT SEGMENTER")
print("=" * 60)

model = JTSegmenter(model_name=MODEL_NAME,
                    num_labels=NUM_LABELS,
                    dropout=DROPOUT).to(DEVICE)

# Parameter counts
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
encoder_params   = sum(p.numel() for p in model.encoder.parameters())
head_params      = (sum(p.numel() for p in model.classifier.parameters())
                    + sum(p.numel() for p in model.crf.parameters()))

print(f"Encoder    : {MODEL_NAME}")
print(f"Hidden size: {model.config.hidden_size}")
print(f"Num labels : {NUM_LABELS}")
print(f"Dropout    : {DROPOUT}")
print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable_params:,}")
print(f"  └─ Encoder (XLM-R-base) : {encoder_params:,}")
print(f"  └─ Linear + CRF head    : {head_params:,}")

# CRF transition / start / end matrices (sanity peek)
print("\nCRF parameters (initial, before training):")
print(f"  start_transitions : {model.crf.start_transitions.detach().cpu().numpy()}")
print(f"  end_transitions   : {model.crf.end_transitions.detach().cpu().numpy()}")
print(f"  transitions (2x2) :\n{model.crf.transitions.detach().cpu().numpy()}")


# ============================================================
# Forward-pass smoke tests
# ============================================================
print("\n" + "=" * 60)
print("SMOKE TEST — training forward")
print("=" * 60)

model.train()
batch = next(iter(peek_loader))
batch_gpu = {k: v.to(DEVICE) for k, v in batch.items() if isinstance(v, torch.Tensor)}

with torch.no_grad():
    out = model(input_ids=batch_gpu["input_ids"],
                attention_mask=batch_gpu["attention_mask"],
                labels=batch_gpu["labels"])

print(f"loss        : {out['loss'].item():.4f}")
print(f"emissions   : {tuple(out['emissions'].shape)}")
# Expected initial loss for 2-class CRF over ~1100 valid tokens: ~0.69 * mean_len
# (model is roughly chance until trained). A finite, positive value is enough.
assert torch.isfinite(out["loss"]), "Loss is NaN/Inf!"

print("\n" + "=" * 60)
print("SMOKE TEST — inference forward")
print("=" * 60)

model.eval()
with torch.no_grad():
    out = model(input_ids=batch_gpu["input_ids"],
                attention_mask=batch_gpu["attention_mask"],
                labels=None)

preds = out["predictions"]
print(f"# decoded sequences : {len(preds)}")
print(f"first sequence len  : {len(preds[0])}  "
      f"(matches attention_mask sum: {int(batch_gpu['attention_mask'][0].sum().item())})")
print(f"first 30 predicted tags : {preds[0][:30]}")

# At init, CRF will tend to predict all-0 (majority class). That's fine.
n_pos_pred = sum(sum(1 for t in seq if t == 1) for seq in preds)
n_total    = sum(len(seq) for seq in preds)
print(f"predicted positives : {n_pos_pred} / {n_total}  "
      f"(expect mostly 0s before training)")

# VRAM check
if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)
    print(f"\nGPU memory: allocated={alloc:.2f} GB | reserved={reserved:.2f} GB")

# Cleanup
del out, batch_gpu
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n✅ Cell 4 complete — JT Segmenter ready for Cell 5 (training loop).")

BUILDING JT SEGMENTER
Encoder    : xlm-roberta-base
Hidden size: 768
Num labels : 2
Dropout    : 0.1
Total params     : 278,045,194
Trainable params : 278,045,194
  └─ Encoder (XLM-R-base) : 278,043,648
  └─ Linear + CRF head    : 1,546

CRF parameters (initial, before training):
  start_transitions : [0.00185589 0.03580422]
  end_transitions   : [ 0.05753712 -0.04360517]
  transitions (2x2) :
[[-0.03318169  0.06633059]
 [ 0.07159882  0.02369589]]

SMOKE TEST — training forward
loss        : 76.0446
emissions   : (16, 256, 2)

SMOKE TEST — inference forward
# decoded sequences : 16
first sequence len  : 256  (matches attention_mask sum: 256)
first 30 predicted tags : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
predicted positives : 2008 / 2008  (expect mostly 0s before training)

GPU memory: allocated=1.04 GB | reserved=2.17 GB

✅ Cell 4 complete — JT Segmenter ready for Cell 5 (training loop).


### Cell 5 — Training Loop (AdamW + Warmup + Mixed Precision)

In [46]:
# ============================================================
# Cell 5: Training loop — AdamW + linear warmup + AMP + best-ckpt saving
# ============================================================

import time
import math
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# ---------- Bump training batch size ----------
BATCH_SIZE_TRAIN = 24
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE_TRAIN,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn,
    drop_last=False,
)
print(f"train_loader rebuilt: {len(train_ds):,} chunks → {len(train_loader):,} "
      f"batches (batch_size={BATCH_SIZE_TRAIN})")

# ---------- Training hyperparameters ----------
NUM_EPOCHS         = 5
LR_ENCODER         = 2e-5      # XLM-R encoder
LR_HEAD            = 5e-4      # linear + CRF head (much smaller, can train faster)
WEIGHT_DECAY       = 0.01
WARMUP_RATIO       = 0.1       # 10% of total steps for linear warmup
MAX_GRAD_NORM      = 1.0
LOG_EVERY          = 100       # log training loss every N batches
USE_AMP            = True      # mixed precision on CUDA

CKPT_DIR = Path("checkpoints")
CKPT_DIR.mkdir(exist_ok=True)
BEST_CKPT_PATH = CKPT_DIR / "jt_segmenter_best.pt"

# ============================================================
# Optimizer — split parameter groups
# ============================================================
# (1) encoder params with weight decay (excluding bias / LayerNorm)
# (2) encoder params without weight decay
# (3) head params with weight decay
# (4) head params without weight decay
no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")

encoder_params = list(model.encoder.named_parameters())
head_params    = (list(model.classifier.named_parameters())
                  + [(f"crf.{n}", p) for n, p in model.crf.named_parameters()])

optim_groups = [
    {
        "params": [p for n, p in encoder_params if not any(nd in n for nd in no_decay)],
        "weight_decay": WEIGHT_DECAY,
        "lr": LR_ENCODER,
    },
    {
        "params": [p for n, p in encoder_params if     any(nd in n for nd in no_decay)],
        "weight_decay": 0.0,
        "lr": LR_ENCODER,
    },
    {
        "params": [p for n, p in head_params    if not any(nd in n for nd in no_decay)],
        "weight_decay": WEIGHT_DECAY,
        "lr": LR_HEAD,
    },
    {
        "params": [p for n, p in head_params    if     any(nd in n for nd in no_decay)],
        "weight_decay": 0.0,
        "lr": LR_HEAD,
    },
]

optimizer = AdamW(optim_groups, betas=(0.9, 0.999), eps=1e-8)

total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"Total optimization steps : {total_steps:,}")
print(f"Warmup steps             : {warmup_steps:,} ({WARMUP_RATIO * 100:.0f}%)")
print(f"LR (encoder / head)      : {LR_ENCODER} / {LR_HEAD}")
print(f"Weight decay             : {WEIGHT_DECAY}")
print(f"AMP (mixed precision)    : {USE_AMP and torch.cuda.is_available()}")

# AMP scaler (only used on CUDA)
scaler = torch.amp.GradScaler('cuda', enabled=(USE_AMP and torch.cuda.is_available()))


# ============================================================
# Validation — token-level F1 on positive class (boundary)
# ============================================================
@torch.no_grad()
def evaluate_loader(model, loader, device, label_pad=LABEL_PAD):
    """
    Token-level evaluation on a loader.
    Computes precision / recall / F1 for the positive (boundary) class,
    over first-subword positions only (i.e., where labels != -100).
    Predictions come from CRF Viterbi decode over the full sequence;
    we read off only the positions whose gold label != -100.
    Also returns total loss for monitoring.
    """
    model.eval()
    tp = fp = fn = tn = 0
    total_loss = 0.0
    n_batches  = 0

    for batch in loader:
        input_ids      = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels         = batch["labels"].to(device, non_blocking=True)

        # Loss
        out = model(input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels)
        total_loss += out["loss"].item()
        n_batches  += 1

        # Predictions (CRF decode over full attention-mask region)
        out_pred = model(input_ids=input_ids,
                         attention_mask=attention_mask,
                         labels=None)
        preds = out_pred["predictions"]   # list[list[int]], length B

        labels_cpu = labels.cpu().tolist()
        for i, seq_pred in enumerate(preds):
            gold_row = labels_cpu[i]
            # Iterate over scoring positions only
            for t, gold in enumerate(gold_row):
                if gold == label_pad:
                    continue
                if t >= len(seq_pred):
                    break
                pred = seq_pred[t]
                if   gold == 1 and pred == 1: tp += 1
                elif gold == 0 and pred == 1: fp += 1
                elif gold == 1 and pred == 0: fn += 1
                else:                         tn += 1

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) else 0.0)
    accuracy  = (tp + tn) / max(1, tp + fp + fn + tn)
    avg_loss  = total_loss / max(1, n_batches)

    return {
        "loss":      avg_loss,
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
        "accuracy":  accuracy,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    }


# ============================================================
# Training loop
# ============================================================
def format_seconds(s):
    m, s = divmod(int(s), 60)
    h, m = divmod(m, 60)
    return f"{h:d}h{m:02d}m{s:02d}s" if h else f"{m:d}m{s:02d}s"

print("\n" + "=" * 60)
print("STARTING TRAINING")
print("=" * 60)

best_val_f1 = -1.0
history     = []         # list of dicts, one per epoch
global_step = 0
t_train_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    t_epoch = time.time()
    running_loss = 0.0
    running_n    = 0
    log_loss     = 0.0
    log_n        = 0

    for step, batch in enumerate(train_loader, start=1):
        input_ids      = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels         = batch["labels"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda',
                                enabled=(USE_AMP and torch.cuda.is_available())):
            out  = model(input_ids=input_ids,
                         attention_mask=attention_mask,
                         labels=labels)
            loss = out["loss"]

        if USE_AMP and torch.cuda.is_available():
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()

        scheduler.step()
        global_step += 1

        loss_val = loss.item()
        running_loss += loss_val
        running_n    += 1
        log_loss     += loss_val
        log_n        += 1

        if step % LOG_EVERY == 0:
            cur_lr_enc = optimizer.param_groups[0]["lr"]
            cur_lr_hd  = optimizer.param_groups[2]["lr"]
            elapsed    = time.time() - t_epoch
            print(f"  epoch {epoch}  step {step:>4}/{len(train_loader)}  "
                  f"loss={log_loss / log_n:7.3f}  "
                  f"lr_enc={cur_lr_enc:.2e}  lr_head={cur_lr_hd:.2e}  "
                  f"elapsed={format_seconds(elapsed)}")
            log_loss = 0.0
            log_n    = 0

    train_loss = running_loss / max(1, running_n)
    epoch_time = time.time() - t_epoch

    # ---------- Validation ----------
    t_val = time.time()
    val   = evaluate_loader(model, val_loader, DEVICE)
    val_time = time.time() - t_val

    history.append({
        "epoch":      epoch,
        "train_loss": train_loss,
        "val_loss":   val["loss"],
        "val_p":      val["precision"],
        "val_r":      val["recall"],
        "val_f1":     val["f1"],
        "val_acc":    val["accuracy"],
        "epoch_time": epoch_time,
    })

    print(f"\n[Epoch {epoch}/{NUM_EPOCHS}]  "
          f"train_loss={train_loss:.4f}  "
          f"val_loss={val['loss']:.4f}  "
          f"val_P={val['precision']:.4f}  "
          f"val_R={val['recall']:.4f}  "
          f"val_F1={val['f1']:.4f}  "
          f"val_acc={val['accuracy']:.4f}")
    print(f"           tp={val['tp']}  fp={val['fp']}  fn={val['fn']}  tn={val['tn']}  "
          f"epoch_time={format_seconds(epoch_time)}  val_time={format_seconds(val_time)}")

    # ---------- Save best ----------
    if val["f1"] > best_val_f1:
        best_val_f1 = val["f1"]
        torch.save({
            "epoch":          epoch,
            "model_state":    model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "best_val_f1":    best_val_f1,
            "val_metrics":    val,
            "config": {
                "model_name": MODEL_NAME,
                "num_labels": NUM_LABELS,
                "max_len":    MAX_LEN,
                "stride":     STRIDE,
                "dropout":    DROPOUT,
            },
        }, BEST_CKPT_PATH)
        print(f"           ✓ new best F1 — saved to {BEST_CKPT_PATH}")

    print()

t_train_total = time.time() - t_train_start
print("=" * 60)
print(f"TRAINING COMPLETE — total time: {format_seconds(t_train_total)}")
print(f"Best validation F1: {best_val_f1:.4f}")
print(f"Best checkpoint saved at: {BEST_CKPT_PATH}")
print("=" * 60)

# ---------- History summary ----------
print("\nEpoch summary:")
print(f"{'Ep':>2} | {'train_loss':>10} | {'val_loss':>9} | {'val_P':>6} | "
      f"{'val_R':>6} | {'val_F1':>6} | {'time':>9}")
print("-" * 70)
for h in history:
    print(f"{h['epoch']:>2} | {h['train_loss']:>10.4f} | {h['val_loss']:>9.4f} | "
          f"{h['val_p']:>6.4f} | {h['val_r']:>6.4f} | {h['val_f1']:>6.4f} | "
          f"{format_seconds(h['epoch_time']):>9}")

print("\n✅ Cell 5 complete — best checkpoint saved, ready for Cell 6 "
      "(load best ckpt + paragraph-level reassembly + post-processor + test set evaluation).")

train_loader rebuilt: 57,218 chunks → 2,385 batches (batch_size=24)
Total optimization steps : 11,925
Warmup steps             : 1,192 (10%)
LR (encoder / head)      : 2e-05 / 0.0005
Weight decay             : 0.01
AMP (mixed precision)    : True

STARTING TRAINING
  epoch 1  step  100/2385  loss= 46.307  lr_enc=1.68e-06  lr_head=4.19e-05  elapsed=0m55s
  epoch 1  step  200/2385  loss=  5.122  lr_enc=3.36e-06  lr_head=8.39e-05  elapsed=1m51s
  epoch 1  step  300/2385  loss=  1.507  lr_enc=5.03e-06  lr_head=1.26e-04  elapsed=2m47s
  epoch 1  step  400/2385  loss=  1.093  lr_enc=6.71e-06  lr_head=1.68e-04  elapsed=3m42s
  epoch 1  step  500/2385  loss=  0.858  lr_enc=8.39e-06  lr_head=2.10e-04  elapsed=4m37s
  epoch 1  step  600/2385  loss=  0.724  lr_enc=1.01e-05  lr_head=2.52e-04  elapsed=5m31s
  epoch 1  step  700/2385  loss=  0.634  lr_enc=1.17e-05  lr_head=2.94e-04  elapsed=6m26s
  epoch 1  step  800/2385  loss=  0.604  lr_enc=1.34e-05  lr_head=3.36e-04  elapsed=7m21s
  epoch 1  ste

## Cell R — Resume from Restart (run this first)

In [1]:
# ============================================================
# Cell R: Bootstrap — resume after kernel restart
# Re-runs imports, environment, data pipeline, encoders, model,
# then loads the best checkpoint from disk.
# ============================================================

import os, re, sys, json, random, warnings, unicodedata, time, math
from pathlib import Path
from collections import Counter, defaultdict
from difflib import SequenceMatcher

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig
from torchcrf import CRF

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} | PyTorch {torch.__version__}")

# ---------- Paths & constants ----------
DATA_DIR    = Path("data")
CKPT_DIR    = Path("checkpoints")
BEST_CKPT_PATH = CKPT_DIR / "jt_segmenter_best.pt"
assert DATA_DIR.exists(),     f"Missing {DATA_DIR.resolve()}"
assert BEST_CKPT_PATH.exists(), f"Missing {BEST_CKPT_PATH.resolve()}"

QUOTE_COMMA = {'"', ',', "'", '\u201c', '\u201d', '\u2018', '\u2019'}

MODEL_NAME       = "xlm-roberta-base"
NUM_LABELS       = 2
DROPOUT          = 0.1
MAX_LEN          = 256
STRIDE           = 64
LABEL_PAD        = -100
BATCH_SIZE_EVAL  = 32
NUM_WORKERS      = 0
PIN_MEMORY       = True


# ============================================================
# Verified data-pipeline functions (Cell 1)
# ============================================================
def parse_paired_files(unseg_path, seg_path):
    with open(unseg_path, "r", encoding="utf-8") as f:
        unseg_lines = [ln.rstrip("\n") for ln in f if ln.strip()]
    paragraphs = []
    for ln in unseg_lines:
        s = ln.strip()
        if s.startswith("<p>"):  s = s[3:]
        if s.endswith("</p>"):   s = s[:-4]
        paragraphs.append(s.strip())

    with open(seg_path, "r", encoding="utf-8") as f:
        seg_lines = [ln.rstrip("\n") for ln in f if ln.strip()]

    grouped, current, in_para = [], [], False
    for ln in seg_lines:
        s = ln.strip()
        starts, ends = s.startswith("<p>"), s.endswith("</p>")
        if starts: s = s[3:]
        if ends:   s = s[:-4]
        s = s.strip()
        if starts:
            current = [s]; in_para = True
            if ends:
                grouped.append(current); current = []; in_para = False
        elif ends:
            current.append(s); grouped.append(current); current = []; in_para = False
        else:
            (current if in_para else grouped).append(s) if not in_para else current.append(s)
    if len(paragraphs) != len(grouped):
        raise ValueError(f"Paragraph count mismatch: {len(paragraphs)} vs {len(grouped)}")
    return [{"pid": i, "paragraph": p, "sentences": [x for x in g if x]}
            for i, (p, g) in enumerate(zip(paragraphs, grouped))]


def _tokenize_ws(text): return text.split()

def _strip_quote_comma(tok):
    out = tok; changed = True
    while changed and out:
        changed = False
        if out and out[0]  in QUOTE_COMMA: out = out[1:];  changed = True
        if out and out[-1] in QUOTE_COMMA: out = out[:-1]; changed = True
    return out

def _fuzzy_tail_match(para_tokens, start_idx, sent_tokens, max_extra=3):
    n_sent = len(sent_tokens)
    if n_sent == 0: return None
    for extra in range(0, max_extra + 1):
        end_idx = start_idx + n_sent - 1 + extra
        if end_idx >= len(para_tokens): break
        para_last = _strip_quote_comma(para_tokens[end_idx])
        sent_last = _strip_quote_comma(sent_tokens[-1])
        if para_last == sent_last and para_last != "":
            span = " ".join(para_tokens[start_idx:end_idx + 1])
            ref  = " ".join(sent_tokens)
            if SequenceMatcher(None, span, ref).ratio() >= 0.80:
                return end_idx
    return None

def label_paragraph_v5(paragraph, sentences):
    para_tokens = _tokenize_ws(paragraph)
    n = len(para_tokens); labels = [0] * n; cursor = 0
    for sent in sentences:
        st = _tokenize_ws(sent)
        if not st: continue
        end_idx = _fuzzy_tail_match(para_tokens, cursor, st, max_extra=3)
        if end_idx is None:
            end_idx = cursor + len(st) - 1
            if end_idx >= n: end_idx = n - 1
        while end_idx > cursor and _strip_quote_comma(para_tokens[end_idx]) == "":
            end_idx -= 1
        labels[end_idx] = 1
        cursor = end_idx + 1
    if n > 0 and labels[-1] == 0:
        last_real = n - 1
        while last_real > 0 and _strip_quote_comma(para_tokens[last_real]) == "":
            last_real -= 1
        for i in range(last_real, n): labels[i] = 0
        labels[last_real] = 1
    return para_tokens, labels

def build_labeled_dataset_v5(aligned, verbose=True, name=""):
    out, n_tok, n_b, n_fail = [], 0, 0, 0
    for item in aligned:
        try:
            tokens, labels = label_paragraph_v5(item["paragraph"], item["sentences"])
            if len(tokens) != len(labels): raise ValueError("len mismatch")
            out.append({"pid": item["pid"], "tokens": tokens,
                        "labels": labels, "n_sent": len(item["sentences"])})
            n_tok += len(tokens); n_b += sum(labels)
        except Exception as e:
            n_fail += 1
    if verbose:
        ratio = (n_b / n_tok * 100) if n_tok else 0.0
        print(f"[{name}] paragraphs={len(out):,} | tokens={n_tok:,} | "
              f"boundaries={n_b:,} ({ratio:.2f}%) | failures={n_fail}")
    return out


# ============================================================
# Load all splits
# ============================================================
print("\nLoading splits...")
train_aligned = parse_paired_files(DATA_DIR / "training_unseg.txt",
                                   DATA_DIR / "training_seg.txt")
train_data = build_labeled_dataset_v5(train_aligned, name="train")
val_aligned = parse_paired_files(DATA_DIR / "validation_unseg.txt",
                                 DATA_DIR / "validation_seg.txt")
val_data = build_labeled_dataset_v5(val_aligned, name="val")
test_data = {}
for i in range(1, 6):
    al = parse_paired_files(DATA_DIR / f"test{i}_unseg.txt",
                            DATA_DIR / f"test{i}_seg.txt")
    test_data[f"test{i}"] = build_labeled_dataset_v5(al, name=f"test{i}")


# ============================================================
# Tokenizer + chunked encoding (Cell 2)
# ============================================================
print("\nLoading tokenizer & encoding splits...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
PAD_ID = tokenizer.pad_token_id

def encode_paragraph(tokens, labels, tokenizer,
                     max_len=MAX_LEN, stride=STRIDE, label_pad=LABEL_PAD):
    enc = tokenizer(tokens, is_split_into_words=True, truncation=False,
                    add_special_tokens=False)
    sub_ids  = enc["input_ids"]
    word_ids = enc.word_ids()
    sub_labels = []; prev = None
    for w in word_ids:
        if w is None:        sub_labels.append(label_pad)
        elif w != prev:      sub_labels.append(labels[w]); prev = w
        else:                sub_labels.append(label_pad)
    cls_id, sep_id = tokenizer.cls_token_id, tokenizer.sep_token_id
    inner_max = max_len - 2
    if len(sub_ids) <= inner_max:
        ranges = [(0, len(sub_ids))]
    else:
        ranges = []; start = 0
        while start < len(sub_ids):
            end = min(start + inner_max, len(sub_ids))
            ranges.append((start, end))
            if end == len(sub_ids): break
            start = end - stride
    chunks = []
    for ci, (s, e) in enumerate(ranges):
        ids  = [cls_id] + sub_ids[s:e]   + [sep_id]
        labs = [label_pad] + sub_labels[s:e] + [label_pad]
        wids = [None] + word_ids[s:e]    + [None]
        chunks.append({"input_ids": ids, "attention_mask": [1]*len(ids),
                       "labels": labs, "word_ids": wids,
                       "chunk_idx": ci, "n_chunks": len(ranges)})
    return chunks

def build_encoded_dataset(labeled, tokenizer, name="",
                          max_len=MAX_LEN, stride=STRIDE):
    flat = []; n_long = 0
    for item in labeled:
        chunks = encode_paragraph(item["tokens"], item["labels"], tokenizer,
                                  max_len=max_len, stride=stride)
        if len(chunks) > 1: n_long += 1
        for ch in chunks:
            ch["pid"] = item["pid"]; ch["n_words"] = len(item["tokens"])
            flat.append(ch)
    if name:
        print(f"[{name}] chunks={len(flat):,} | long-paragraphs={n_long:,}")
    return flat

train_enc = build_encoded_dataset(train_data, tokenizer, name="train")
val_enc   = build_encoded_dataset(val_data,   tokenizer, name="val")
test_enc  = {k: build_encoded_dataset(v, tokenizer, name=k)
             for k, v in test_data.items()}


# ============================================================
# Dataset / collator / loaders (Cell 3)
# ============================================================
class MizoSegDataset(Dataset):
    def __init__(self, ec): self.data = ec
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        ch = self.data[i]
        return {"input_ids": ch["input_ids"],
                "attention_mask": ch["attention_mask"],
                "labels": ch["labels"], "word_ids": ch["word_ids"],
                "pid": ch["pid"], "chunk_idx": ch["chunk_idx"],
                "n_chunks": ch["n_chunks"], "n_words": ch["n_words"]}

def collate_fn(batch):
    max_len = max(len(b["input_ids"]) for b in batch)
    iid, am, lb, meta = [], [], [], []
    for b in batch:
        L = len(b["input_ids"]); pad = max_len - L
        iid.append(b["input_ids"]      + [PAD_ID]    * pad)
        am.append( b["attention_mask"] + [0]         * pad)
        lb.append( b["labels"]         + [LABEL_PAD] * pad)
        meta.append({"word_ids": b["word_ids"] + [None]*pad,
                     "pid": b["pid"], "chunk_idx": b["chunk_idx"],
                     "n_chunks": b["n_chunks"], "n_words": b["n_words"],
                     "valid_len": L})
    return {"input_ids":      torch.tensor(iid, dtype=torch.long),
            "attention_mask": torch.tensor(am,  dtype=torch.long),
            "labels":         torch.tensor(lb,  dtype=torch.long),
            "meta": meta}

train_ds = MizoSegDataset(train_enc)
val_ds   = MizoSegDataset(val_enc)
test_ds  = {k: MizoSegDataset(v) for k, v in test_enc.items()}

val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE_EVAL, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                        collate_fn=collate_fn)
test_loaders = {k: DataLoader(ds, batch_size=BATCH_SIZE_EVAL, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                              collate_fn=collate_fn)
                for k, ds in test_ds.items()}


# ============================================================
# Model definition (Cell 4) + load checkpoint
# ============================================================
class JTSegmenter(nn.Module):
    def __init__(self, model_name=MODEL_NAME, num_labels=NUM_LABELS, dropout=DROPOUT):
        super().__init__()
        self.config     = AutoConfig.from_pretrained(model_name)
        self.encoder    = AutoModel.from_pretrained(model_name)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)
        self.crf        = CRF(num_labels, batch_first=True)
        self.num_labels = num_labels

    @staticmethod
    def _pack_for_crf(emissions, labels):
        B, T, C = emissions.shape
        valid   = labels.ne(-100)
        lengths = valid.sum(dim=1)
        max_len = int(lengths.max().item())
        device  = emissions.device
        pe = torch.zeros(B, max_len, C, device=device, dtype=emissions.dtype)
        pl = torch.zeros(B, max_len,    device=device, dtype=torch.long)
        pm = torch.zeros(B, max_len,    device=device, dtype=torch.bool)
        for i in range(B):
            n = int(lengths[i].item())
            if n == 0:
                pm[i, 0] = True; continue
            sel = valid[i].nonzero(as_tuple=False).squeeze(1)
            pe[i, :n] = emissions[i, sel]
            pl[i, :n] = labels[i, sel]
            pm[i, :n] = True
        return pe, pl, pm, lengths

    def forward(self, input_ids, attention_mask, labels=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden    = self.dropout(out.last_hidden_state)
        emissions = self.classifier(hidden)
        if labels is not None:
            pe, pl, pm, _ = self._pack_for_crf(emissions, labels)
            log_lik = self.crf(pe, pl, mask=pm, reduction="mean")
            return {"loss": -log_lik, "emissions": emissions}
        crf_mask = attention_mask.bool()
        decoded  = self.crf.decode(emissions, mask=crf_mask)
        return {"predictions": decoded, "emissions": emissions}


print("\nBuilding model & loading checkpoint...")
model = JTSegmenter().to(DEVICE)
ckpt = torch.load(BEST_CKPT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"✓ Loaded epoch {ckpt['epoch']} | "
      f"val_F1={ckpt['best_val_f1']:.4f} | "
      f"val_P={ckpt['val_metrics']['precision']:.4f} | "
      f"val_R={ckpt['val_metrics']['recall']:.4f}")

print("\n✅ Resume complete — all state restored. You can now run Cell 6 from "
      "the section starting at '6.2 Paragraph-level prediction reassembly'.")

C:\Users\Haulai\miniconda3\envs\mizen\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Device: cuda | PyTorch 2.7.1+cu118

Loading splits...
[train] paragraphs=46,909 | tokens=3,069,899 | boundaries=181,348 (5.91%) | failures=0
[val] paragraphs=500 | tokens=35,732 | boundaries=2,087 (5.84%) | failures=0
[test1] paragraphs=100 | tokens=7,244 | boundaries=422 (5.83%) | failures=0
[test2] paragraphs=100 | tokens=6,072 | boundaries=363 (5.98%) | failures=0
[test3] paragraphs=100 | tokens=6,809 | boundaries=412 (6.05%) | failures=0
[test4] paragraphs=100 | tokens=7,810 | boundaries=438 (5.61%) | failures=0
[test5] paragraphs=100 | tokens=5,965 | boundaries=374 (6.27%) | failures=0

Loading tokenizer & encoding splits...


Token indices sequence length is longer than the specified maximum sequence length for this model (621 > 512). Running this sequence through the model will result in indexing errors


[train] chunks=57,218 | long-paragraphs=6,829
[val] chunks=634 | long-paragraphs=86
[test1] chunks=131 | long-paragraphs=17
[test2] chunks=117 | long-paragraphs=15
[test3] chunks=124 | long-paragraphs=13
[test4] chunks=131 | long-paragraphs=16
[test5] chunks=115 | long-paragraphs=12

Building model & loading checkpoint...
✓ Loaded epoch 5 | val_F1=0.9921 | val_P=0.9902 | val_R=0.9940

✅ Resume complete — all state restored. You can now run Cell 6 from the section starting at '6.2 Paragraph-level prediction reassembly'.


### Cell 6 — Inference, Post-Processor & Test-Set Evaluation

In [2]:
# ============================================================
# Cell 6: Load best ckpt, reassemble predictions across chunks,
#         apply Mizo post-processor, evaluate on all 5 test sets
# ============================================================

# ============================================================
# 6.1 Load best checkpoint
# ============================================================
print("=" * 60)
print(f"LOADING BEST CHECKPOINT: {BEST_CKPT_PATH}")
print("=" * 60)

ckpt = torch.load(BEST_CKPT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"Loaded epoch {ckpt['epoch']} | "
      f"val_F1={ckpt['best_val_f1']:.4f} | "
      f"val_P={ckpt['val_metrics']['precision']:.4f} | "
      f"val_R={ckpt['val_metrics']['recall']:.4f}")


# ============================================================
# 6.2 Paragraph-level prediction reassembly
# ============================================================
@torch.no_grad()
def predict_paragraphs(model, loader, device):
    """
    Run inference over a loader (which may contain multiple chunks per
    paragraph due to long-paragraph chunking) and reassemble word-level
    predictions for each paragraph.

    Strategy for overlapping chunks:
      For each (pid, word_index), collect all subword-level predictions
      from any chunk that scored that word's first subword. If a word is
      covered by multiple chunks (overlap region), majority-vote; ties
      break toward 1 (recall-friendly — the post-processor will trim
      false positives).

    Returns: dict[pid] -> list[int]  (word-level predictions, len = n_words)
    """
    model.eval()
    # word_preds[pid][word_idx] = list[int] of votes from all chunks
    word_votes = defaultdict(lambda: defaultdict(list))
    n_words_by_pid = {}

    for batch in loader:
        input_ids      = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)

        out = model(input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=None)
        preds = out["predictions"]   # list[list[int]]

        for i, seq_pred in enumerate(preds):
            meta = batch["meta"][i]
            pid       = meta["pid"]
            n_words   = meta["n_words"]
            word_ids  = meta["word_ids"]
            valid_len = meta["valid_len"]
            n_words_by_pid[pid] = n_words

            # First-subword detection: only the first subword of each word
            # has a meaningful prediction position. Iterate up to valid_len
            # (the unpadded chunk length).
            prev_w = None
            for t in range(min(valid_len, len(seq_pred))):
                w = word_ids[t]
                if w is None:
                    prev_w = None
                    continue
                if w != prev_w:
                    word_votes[pid][w].append(seq_pred[t])
                    prev_w = w

    # Resolve votes
    predictions = {}
    for pid, n_words in n_words_by_pid.items():
        row = [0] * n_words
        votes = word_votes[pid]
        for w in range(n_words):
            v = votes.get(w, [])
            if not v:
                # Defensive: word never scored (shouldn't happen with stride)
                row[w] = 0
            else:
                ones = sum(v)
                zeros = len(v) - ones
                # Tie → 1 (recall-friendly). Strict majority otherwise.
                row[w] = 1 if ones >= zeros and ones > 0 else 0
        predictions[pid] = row
    return predictions


# ============================================================
# 6.3 Mizo post-processor — comma rules for `a,` and `va,`
# ============================================================
def apply_mizo_postprocessor(tokens, preds, verbose=False):
    """
    Post-process binary predictions using Mizo-specific comma rules.

    Two patterns in Mizo discourse where the segmenter tends to either
    over- or under-segment:

      RULE 1 — `a,` / `va,` clause-linker:
        When a token ends in `a,` or `va,` (lowercase, comma attached),
        this is a non-final clause linker ("and then …" / "is, and …").
        It should NEVER be a sentence boundary — force prediction to 0.
        Even if the model predicted 1, demote to 0.

      RULE 2 — protection of `a.` / `va.` / `ni.` boundaries:
        Tokens ending in `.` (full stop) at the end of a paragraph-
        internal segment are almost always sentence boundaries. If the
        model predicted 0 on a clear sentence-final marker (`a.`, `va.`,
        `ni.`, or any token ending in `.`), promote to 1.

    Returns: post-processed predictions (list[int]) and a small dict of
             counts for diagnostics.
    """
    n = len(tokens)
    out = list(preds)
    n_demoted  = 0   # 1 → 0  (RULE 1)
    n_promoted = 0   # 0 → 1  (RULE 2)

    for i, tok in enumerate(tokens):
        low = tok.lower()
        # RULE 1: `a,` / `va,` linker — never a boundary
        if low.endswith("a,") or low.endswith("va,"):
            if out[i] == 1:
                out[i] = 0
                n_demoted += 1
            continue

        # RULE 2: token ending in `.` is almost always sentence-final.
        # Be conservative: only promote when the token is a clear Mizo
        # sentence-final particle (`a.`, `va.`, `ni.`, `e.`) — a blanket
        # "any '.' = boundary" rule would damage abbreviations.
        if out[i] == 0:
            if low in {"a.", "va.", "ni.", "e."}:
                out[i] = 1
                n_promoted += 1

    # Always ensure last token is a boundary (paragraph ends a sentence)
    if n > 0 and out[-1] == 0:
        # find last non-quote/comma artefact
        last = n - 1
        while last > 0 and _strip_quote_comma(tokens[last]) == "":
            last -= 1
        for j in range(last, n):
            out[j] = 0
        out[last] = 1

    return out, {"demoted": n_demoted, "promoted": n_promoted}


# ============================================================
# 6.4 Evaluation function — token-level + sentence-level metrics
# ============================================================
def evaluate_split(model, loader, gold_data, device,
                   apply_pp=True, name=""):
    """
    Run inference, reassemble paragraph-level predictions, optionally apply
    the Mizo post-processor, and compute:
      - Token-level: precision / recall / F1 / accuracy on positive class.
      - Sentence-level: a sentence is correctly segmented iff its set of
        boundary positions exactly matches gold (within the paragraph).
        We report % paragraphs perfectly segmented.
    """
    preds_by_pid = predict_paragraphs(model, loader, device)

    tp = fp = fn = tn = 0
    perfect = 0
    pp_demoted = pp_promoted = 0

    per_para = []   # for later inspection
    for item in gold_data:
        pid    = item["pid"]
        tokens = item["tokens"]
        gold   = item["labels"]
        pred   = preds_by_pid.get(pid, [0] * len(tokens))

        if apply_pp:
            pred_pp, ppstats = apply_mizo_postprocessor(tokens, pred)
            pp_demoted  += ppstats["demoted"]
            pp_promoted += ppstats["promoted"]
            pred_use = pred_pp
        else:
            pred_use = pred

        # Token-level
        for g, p in zip(gold, pred_use):
            if   g == 1 and p == 1: tp += 1
            elif g == 0 and p == 1: fp += 1
            elif g == 1 and p == 0: fn += 1
            else:                   tn += 1

        # Sentence-level (paragraph-perfect)
        if pred_use == gold:
            perfect += 1

        per_para.append({"pid": pid, "tokens": tokens,
                         "gold": gold, "pred_raw": pred, "pred_pp": pred_use})

    P = tp / (tp + fp) if (tp + fp) else 0.0
    R = tp / (tp + fn) if (tp + fn) else 0.0
    F1 = (2 * P * R / (P + R)) if (P + R) else 0.0
    acc = (tp + tn) / max(1, tp + fp + fn + tn)
    para_acc = perfect / max(1, len(gold_data))

    label = name or "split"
    pp_tag = "  [+ post-processor]" if apply_pp else "  [raw model output]"
    print(f"[{label}]{pp_tag}")
    print(f"  Token-level     : P={P:.4f}  R={R:.4f}  F1={F1:.4f}  acc={acc:.4f}")
    print(f"                    tp={tp}  fp={fp}  fn={fn}  tn={tn}")
    print(f"  Paragraph-exact : {perfect}/{len(gold_data)}  ({para_acc * 100:.1f}%)")
    if apply_pp:
        print(f"  Post-proc edits : demoted(1→0)={pp_demoted}  "
              f"promoted(0→1)={pp_promoted}")
    return {
        "name": label, "P": P, "R": R, "F1": F1, "acc": acc,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "para_exact": perfect, "para_total": len(gold_data),
        "pp_demoted": pp_demoted, "pp_promoted": pp_promoted,
        "per_para": per_para,
    }


# ============================================================
# 6.5 Run — raw and post-processed evaluation on val + 5 test sets
# ============================================================
print("\n" + "=" * 60)
print("VALIDATION SET")
print("=" * 60)
val_raw = evaluate_split(model, val_loader, val_data, DEVICE,
                         apply_pp=False, name="val")
print()
val_pp  = evaluate_split(model, val_loader, val_data, DEVICE,
                         apply_pp=True,  name="val")

print("\n" + "=" * 60)
print("TEST SETS — RAW MODEL OUTPUT")
print("=" * 60)
test_raw_results = {}
for k in ["test1", "test2", "test3", "test4", "test5"]:
    test_raw_results[k] = evaluate_split(
        model, test_loaders[k], test_data[k], DEVICE,
        apply_pp=False, name=k)
    print()

print("=" * 60)
print("TEST SETS — WITH MIZO POST-PROCESSOR")
print("=" * 60)
test_pp_results = {}
for k in ["test1", "test2", "test3", "test4", "test5"]:
    test_pp_results[k] = evaluate_split(
        model, test_loaders[k], test_data[k], DEVICE,
        apply_pp=True, name=k)
    print()


# ============================================================
# 6.6 Aggregate summary table
# ============================================================
print("=" * 60)
print("AGGREGATE SUMMARY — TEST SETS")
print("=" * 60)
print(f"{'Split':<7} | "
      f"{'Raw P':>6}  {'Raw R':>6}  {'Raw F1':>6}  {'Raw P-Ex':>8}  ||  "
      f"{'PP P':>6}  {'PP R':>6}  {'PP F1':>6}  {'PP P-Ex':>8}")
print("-" * 90)
for k in ["test1", "test2", "test3", "test4", "test5"]:
    r = test_raw_results[k]
    p = test_pp_results[k]
    print(f"{k:<7} | "
          f"{r['P']:>6.4f}  {r['R']:>6.4f}  {r['F1']:>6.4f}  "
          f"{r['para_exact']:>3}/{r['para_total']:<4} || "
          f"{p['P']:>6.4f}  {p['R']:>6.4f}  {p['F1']:>6.4f}  "
          f"{p['para_exact']:>3}/{p['para_total']:<4}")

# Macro averages across test sets
def macro(results, key):
    return sum(r[key] for r in results.values()) / len(results)

print("-" * 90)
print(f"{'MACRO':<7} | "
      f"{macro(test_raw_results, 'P'):>6.4f}  "
      f"{macro(test_raw_results, 'R'):>6.4f}  "
      f"{macro(test_raw_results, 'F1'):>6.4f}  "
      f"{'':>8} || "
      f"{macro(test_pp_results, 'P'):>6.4f}  "
      f"{macro(test_pp_results, 'R'):>6.4f}  "
      f"{macro(test_pp_results, 'F1'):>6.4f}")

# Micro averages (pool tp/fp/fn across all test sets)
def micro(results):
    tp = sum(r["tp"] for r in results.values())
    fp = sum(r["fp"] for r in results.values())
    fn = sum(r["fn"] for r in results.values())
    P  = tp / (tp + fp) if (tp + fp) else 0.0
    R  = tp / (tp + fn) if (tp + fn) else 0.0
    F1 = (2 * P * R / (P + R)) if (P + R) else 0.0
    return P, R, F1

mr_P, mr_R, mr_F1 = micro(test_raw_results)
mp_P, mp_R, mp_F1 = micro(test_pp_results)
print(f"{'MICRO':<7} | "
      f"{mr_P:>6.4f}  {mr_R:>6.4f}  {mr_F1:>6.4f}  {'':>8} || "
      f"{mp_P:>6.4f}  {mp_R:>6.4f}  {mp_F1:>6.4f}")

print("\n✅ Cell 6 complete — full evaluation done. "
      "Ready for Cell 7 (BiLSTM-CRF baseline) "
      "or Cell 8 (error analysis + qualitative inspection).")

LOADING BEST CHECKPOINT: checkpoints\jt_segmenter_best.pt
Loaded epoch 5 | val_F1=0.9921 | val_P=0.9902 | val_R=0.9940

VALIDATION SET
[val]  [raw model output]
  Token-level     : P=0.9900  R=0.9962  F1=0.9931  acc=0.9992
                    tp=2079  fp=21  fn=8  tn=33624
  Paragraph-exact : 480/500  (96.0%)

[val]  [+ post-processor]
  Token-level     : P=0.9933  R=0.8491  F1=0.9155  acc=0.9908
                    tp=1772  fp=12  fn=315  tn=33633
  Paragraph-exact : 365/500  (73.0%)
  Post-proc edits : demoted(1→0)=323  promoted(0→1)=5

TEST SETS — RAW MODEL OUTPUT
[test1]  [raw model output]
  Token-level     : P=0.9860  R=1.0000  F1=0.9929  acc=0.9992
                    tp=422  fp=6  fn=0  tn=6816
  Paragraph-exact : 97/100  (97.0%)

[test2]  [raw model output]
  Token-level     : P=1.0000  R=1.0000  F1=1.0000  acc=1.0000
                    tp=363  fp=0  fn=0  tn=5709
  Paragraph-exact : 100/100  (100.0%)

[test3]  [raw model output]
  Token-level     : P=0.9952  R=1.0000  F1=0.9

### Cell 6.5 — Post-Processor Diagnostic

In [4]:
# ============================================================
# Cell 6.5: Diagnose why the post-processor hurts performance
# ============================================================

print("=" * 60)
print("DIAGNOSTIC 1 — Gold boundaries on `a,` / `va,` tokens")
print("=" * 60)

def count_gold_a_comma(data, name):
    n_total_gold   = 0
    n_a_comma_gold = 0
    examples       = []
    for item in data:
        for i, (tok, lab) in enumerate(zip(item["tokens"], item["labels"])):
            if lab == 1:
                n_total_gold += 1
                low = tok.lower()
                if low.endswith("a,") or low.endswith("va,"):
                    n_a_comma_gold += 1
                    if len(examples) < 5:
                        # Context: 3 tokens before, the boundary, 2 after
                        s = max(0, i - 3)
                        e = min(len(item["tokens"]), i + 3)
                        ctx = " ".join(item["tokens"][s:e])
                        examples.append(f"  …{ctx}…  (boundary on {tok!r})")
    pct = n_a_comma_gold / n_total_gold * 100 if n_total_gold else 0
    print(f"[{name}] gold boundaries on a,/va,: "
          f"{n_a_comma_gold:,}/{n_total_gold:,}  ({pct:.2f}%)")
    if examples:
        print(f"  Examples:")
        for ex in examples:
            print(ex)
    return n_a_comma_gold, n_total_gold


count_gold_a_comma(val_data, "val")
print()
for k in ["test1", "test2", "test3", "test4", "test5"]:
    count_gold_a_comma(test_data[k], k)
print()
print("[train] (this is what the model learned from)")
count_gold_a_comma(train_data, "train")


print("\n" + "=" * 60)
print("DIAGNOSTIC 2 — Raw-model errors: top FN / FP tokens")
print("=" * 60)

def collect_errors(model, loader, gold_data, device, name=""):
    preds_by_pid = predict_paragraphs(model, loader, device)
    fn_tokens = Counter()   # gold=1, pred=0  (model missed a boundary)
    fp_tokens = Counter()   # gold=0, pred=1  (model spuriously added one)
    for item in gold_data:
        pid  = item["pid"]
        toks = item["tokens"]
        gold = item["labels"]
        pred = preds_by_pid.get(pid, [0] * len(toks))
        for tok, g, p in zip(toks, gold, pred):
            if   g == 1 and p == 0: fn_tokens[tok] += 1
            elif g == 0 and p == 1: fp_tokens[tok] += 1
    print(f"\n[{name}]")
    print(f"  Top 10 FN tokens (model missed gold boundary):")
    if fn_tokens:
        for tok, n in fn_tokens.most_common(10):
            print(f"    {n:>3} × {tok!r}")
    else:
        print("    (none — perfect recall)")
    print(f"  Top 10 FP tokens (model wrongly predicted boundary):")
    if fp_tokens:
        for tok, n in fp_tokens.most_common(10):
            print(f"    {n:>3} × {tok!r}")
    else:
        print("    (none — perfect precision)")

collect_errors(model, val_loader, val_data, DEVICE, name="val")
for k in ["test1", "test2", "test3", "test4", "test5"]:
    collect_errors(model, test_loaders[k], test_data[k], DEVICE, name=k)

print("\n✅ Diagnostic complete.")

DIAGNOSTIC 1 — Gold boundaries on `a,` / `va,` tokens
[val] gold boundaries on a,/va,: 312/2,087  (14.95%)
  Examples:
  …tak a ni a, Halkha Pawi…  (boundary on 'a,')
  …Sailo a lal a, fanu pali…  (boundary on 'a,')
  …a duh hle a, a duh…  (boundary on 'a,')
  …thei kan ni a,…  (boundary on 'a,')
  …min ti tawh a, kei erawh…  (boundary on 'a,')

[test1] gold boundaries on a,/va,: 65/422  (15.40%)
  Examples:
  …fihlim chuang lo va, kan Aizawl…  (boundary on 'va,')
  …reng bawk si a, inven dan…  (boundary on 'a,')
  …kan lo kai a, khawi khawiah…  (boundary on 'a,')
  …nge an tihdik a, khawi lai…  (boundary on 'a,')
  …chuang lo ang a, mahni inring…  (boundary on 'a,')
[test2] gold boundaries on a,/va,: 48/363  (13.22%)
  Examples:
  …pahnih kan hmu a, pakhat chu…  (boundary on 'a,')
  …pa a ni a, pakhat leh…  (boundary on 'a,')
  …mai a buatsaih a, thu leh…  (boundary on 'a,')
  …ka rilruk deuh a, mahse, ka…  (boundary on 'a,')
  …te a ni a, a then…  (boundary on 'a,')
[test3] gold boun

### Helper that was lost during the kernel restart

In [6]:
# Helper that was lost during the kernel restart
def format_seconds(s):
    m, s = divmod(int(s), 60)
    h, m = divmod(m, 60)
    return f"{h:d}h{m:02d}m{s:02d}s" if h else f"{m:d}m{s:02d}s"

print("✓ format_seconds defined")

✓ format_seconds defined


### Cell 7 — BiLSTM-CRF Baseline

In [7]:
# ============================================================
# Cell 7: BiLSTM-CRF baseline (Char-CNN + Word-Embed → BiLSTM → CRF)
# ============================================================

# ---------- Hyperparameters ----------
WORD_EMB_DIM      = 200
CHAR_EMB_DIM      = 30
CHAR_CNN_FILTERS  = 50          # output channels per filter width
CHAR_CNN_WIDTHS   = (3,)        # single width 3 = standard char-CNN
LSTM_HIDDEN       = 256         # per direction → 512 concatenated
LSTM_LAYERS       = 1
LSTM_DROPOUT      = 0.5         # input/output dropout (Lample 2016 standard)

MIN_WORD_FREQ     = 2           # words with freq < 2 → <unk>
MAX_WORD_LEN      = 25          # truncate long tokens for char-CNN

BLSTM_BATCH_TRAIN = 32
BLSTM_BATCH_EVAL  = 64
BLSTM_EPOCHS      = 10          # BiLSTMs typically need more epochs than transformers
BLSTM_LR          = 1e-3
BLSTM_WD          = 1e-5
BLSTM_WARMUP      = 0.05
BLSTM_GRAD_CLIP   = 5.0

BLSTM_CKPT_PATH   = CKPT_DIR / "bilstm_crf_best.pt"

# ============================================================
# 7.1 Build vocabularies from training data
# ============================================================
print("=" * 60)
print("BUILDING VOCABULARIES FROM TRAINING DATA")
print("=" * 60)

from collections import Counter as _Counter

PAD_W, UNK_W = "<pad>", "<unk>"
PAD_C, UNK_C = "<pad>", "<unk>"

word_counter = _Counter()
char_counter = _Counter()
for item in train_data:
    for tok in item["tokens"]:
        word_counter[tok] += 1
        for ch in tok[:MAX_WORD_LEN]:
            char_counter[ch] += 1

# Word vocab: <pad>=0, <unk>=1, then words with freq >= MIN_WORD_FREQ
word2id = {PAD_W: 0, UNK_W: 1}
for w, c in word_counter.most_common():
    if c >= MIN_WORD_FREQ:
        word2id[w] = len(word2id)

# Char vocab: <pad>=0, <unk>=1, then all chars seen in training
char2id = {PAD_C: 0, UNK_C: 1}
for ch, _ in char_counter.most_common():
    char2id[ch] = len(char2id)

print(f"Word vocab size : {len(word2id):,}  "
      f"(coverage: {sum(1 for w in word_counter if w in word2id):,}"
      f"/{len(word_counter):,} types)")
print(f"Char vocab size : {len(char2id):,}")
print(f"Most common chars : {[c for c, _ in char_counter.most_common(20)]}")


# ============================================================
# 7.2 Encode word-level data for BiLSTM (no subword chunking)
# ============================================================
def encode_for_blstm(tokens, labels):
    """
    Convert (tokens, labels) into:
      - word_ids   : list[int] of length L
      - char_ids   : list[list[int]] of length L, each inner list <= MAX_WORD_LEN
      - labels     : list[int] of length L
    """
    word_ids = [word2id.get(t, word2id[UNK_W]) for t in tokens]
    char_ids = [[char2id.get(c, char2id[UNK_C])
                 for c in t[:MAX_WORD_LEN]]
                for t in tokens]
    return {"word_ids": word_ids,
            "char_ids": char_ids,
            "labels":   list(labels),
            "n_words":  len(tokens)}

print("\nEncoding splits for BiLSTM...")
train_blstm = [{"pid": it["pid"], **encode_for_blstm(it["tokens"], it["labels"])}
               for it in train_data]
val_blstm   = [{"pid": it["pid"], **encode_for_blstm(it["tokens"], it["labels"])}
               for it in val_data]
test_blstm  = {k: [{"pid": it["pid"], **encode_for_blstm(it["tokens"], it["labels"])}
                   for it in v]
               for k, v in test_data.items()}
print(f"train: {len(train_blstm):,} | val: {len(val_blstm):,} | "
      f"test sets: {[len(v) for v in test_blstm.values()]}")


# ============================================================
# 7.3 Dataset & collator (different from transformer collator —
#     handles 2D char tensor padding)
# ============================================================
class BLSTMDataset(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

def collate_blstm(batch):
    """
    Pad to longest paragraph in batch (word dim) and longest word (char dim).
    """
    B = len(batch)
    L_max = max(b["n_words"] for b in batch)
    W_max = max((max((len(c) for c in b["char_ids"]), default=1)
                 for b in batch), default=1)
    W_max = max(W_max, 1)

    word_ids = torch.zeros(B, L_max, dtype=torch.long)         # PAD_W = 0
    char_ids = torch.zeros(B, L_max, W_max, dtype=torch.long)  # PAD_C = 0
    labels   = torch.full((B, L_max), -100, dtype=torch.long)  # ignore-pad
    lengths  = torch.zeros(B, dtype=torch.long)
    pids     = []
    n_words_list = []

    for i, b in enumerate(batch):
        L = b["n_words"]
        word_ids[i, :L] = torch.tensor(b["word_ids"], dtype=torch.long)
        labels[i, :L]   = torch.tensor(b["labels"],   dtype=torch.long)
        lengths[i]      = L
        for j, cs in enumerate(b["char_ids"]):
            if cs:
                char_ids[i, j, :len(cs)] = torch.tensor(cs, dtype=torch.long)
        pids.append(b["pid"])
        n_words_list.append(L)

    return {"word_ids": word_ids, "char_ids": char_ids,
            "labels": labels, "lengths": lengths,
            "pids": pids, "n_words": n_words_list}


train_blstm_loader = DataLoader(BLSTMDataset(train_blstm),
                                batch_size=BLSTM_BATCH_TRAIN, shuffle=True,
                                num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                                collate_fn=collate_blstm)
val_blstm_loader   = DataLoader(BLSTMDataset(val_blstm),
                                batch_size=BLSTM_BATCH_EVAL, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                                collate_fn=collate_blstm)
test_blstm_loaders = {k: DataLoader(BLSTMDataset(v),
                                    batch_size=BLSTM_BATCH_EVAL, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                                    collate_fn=collate_blstm)
                      for k, v in test_blstm.items()}

print(f"train batches: {len(train_blstm_loader):,} | "
      f"val batches: {len(val_blstm_loader)}")


# ============================================================
# 7.4 Model: Char-CNN + Word-Embed → BiLSTM → CRF
# ============================================================
class CharCNN(nn.Module):
    """
    Character-level CNN: embed each char, apply parallel 1D convolutions
    of widths CHAR_CNN_WIDTHS, max-pool each over the word's char dim,
    concatenate. Output dim = CHAR_CNN_FILTERS * len(CHAR_CNN_WIDTHS).
    """
    def __init__(self, vocab_size, emb_dim=CHAR_EMB_DIM,
                 filters=CHAR_CNN_FILTERS, widths=CHAR_CNN_WIDTHS):
        super().__init__()
        self.emb   = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(emb_dim, filters, kernel_size=w, padding=w // 2)
            for w in widths
        ])
        self.out_dim = filters * len(widths)

    def forward(self, char_ids):
        # char_ids: (B, L, W)
        B, L, W = char_ids.shape
        x = self.emb(char_ids.view(B * L, W))           # (B*L, W, E)
        x = x.transpose(1, 2)                           # (B*L, E, W)
        outs = []
        for conv in self.convs:
            h = conv(x)                                 # (B*L, F, W')
            h = torch.relu(h)
            h = h.max(dim=2).values                     # (B*L, F)
            outs.append(h)
        out = torch.cat(outs, dim=1)                    # (B*L, F*nW)
        return out.view(B, L, self.out_dim)             # (B, L, F*nW)


class BiLSTM_CRF(nn.Module):
    def __init__(self, word_vocab_size, char_vocab_size,
                 word_emb=WORD_EMB_DIM, char_emb=CHAR_EMB_DIM,
                 hidden=LSTM_HIDDEN, layers=LSTM_LAYERS,
                 dropout=LSTM_DROPOUT, num_labels=NUM_LABELS):
        super().__init__()
        self.word_emb = nn.Embedding(word_vocab_size, word_emb, padding_idx=0)
        self.char_cnn = CharCNN(char_vocab_size, emb_dim=char_emb)

        self.input_drop = nn.Dropout(dropout)
        in_dim = word_emb + self.char_cnn.out_dim
        self.lstm = nn.LSTM(in_dim, hidden, num_layers=layers,
                            batch_first=True, bidirectional=True,
                            dropout=(dropout if layers > 1 else 0.0))
        self.output_drop = nn.Dropout(dropout)
        self.classifier  = nn.Linear(hidden * 2, num_labels)
        self.crf         = CRF(num_labels, batch_first=True)
        self.num_labels  = num_labels

    def _emit(self, word_ids, char_ids, lengths):
        we = self.word_emb(word_ids)                    # (B, L, Ew)
        ce = self.char_cnn(char_ids)                    # (B, L, F*nW)
        x  = torch.cat([we, ce], dim=-1)
        x  = self.input_drop(x)

        # Pack for LSTM efficiency. Move lengths to CPU as int64.
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        h_packed, _ = self.lstm(packed)
        h, _ = nn.utils.rnn.pad_packed_sequence(h_packed, batch_first=True,
                                                total_length=word_ids.size(1))
        h = self.output_drop(h)
        emissions = self.classifier(h)                  # (B, L, C)
        return emissions

    def forward(self, word_ids, char_ids, lengths, labels=None):
        emissions = self._emit(word_ids, char_ids, lengths)

        # Build CRF mask from real word positions (label != -100 OR within length)
        # Use lengths-based mask, since labels here ARE word-level (no subwords)
        B, L, _ = emissions.shape
        idx = torch.arange(L, device=emissions.device).unsqueeze(0).expand(B, L)
        crf_mask = idx < lengths.unsqueeze(1)           # (B, L) bool

        if labels is not None:
            # CRF needs labels at positions where mask is True; padded positions
            # have label=-100 which would crash CRF. Replace with 0; mask hides them.
            safe_labels = labels.masked_fill(labels == -100, 0)
            log_lik = self.crf(emissions, safe_labels,
                               mask=crf_mask, reduction="mean")
            return {"loss": -log_lik, "emissions": emissions}

        decoded = self.crf.decode(emissions, mask=crf_mask)
        return {"predictions": decoded, "emissions": emissions}


# ============================================================
# 7.5 Build, train
# ============================================================
print("\n" + "=" * 60)
print("BUILDING BiLSTM-CRF")
print("=" * 60)
blstm_model = BiLSTM_CRF(len(word2id), len(char2id)).to(DEVICE)
n_params = sum(p.numel() for p in blstm_model.parameters())
n_trainable = sum(p.numel() for p in blstm_model.parameters() if p.requires_grad)
print(f"Total params     : {n_params:,}")
print(f"Trainable params : {n_trainable:,}")
print(f"  (compare: JT Segmenter = 278,045,194 — BiLSTM is "
      f"{n_params / 278_045_194 * 100:.1f}% the size)")

optimizer_b = torch.optim.AdamW(blstm_model.parameters(),
                                lr=BLSTM_LR, weight_decay=BLSTM_WD)
total_steps_b  = len(train_blstm_loader) * BLSTM_EPOCHS
warmup_steps_b = int(total_steps_b * BLSTM_WARMUP)
from transformers import get_linear_schedule_with_warmup
scheduler_b = get_linear_schedule_with_warmup(
    optimizer_b, num_warmup_steps=warmup_steps_b,
    num_training_steps=total_steps_b)

print(f"Epochs: {BLSTM_EPOCHS} | total_steps: {total_steps_b:,} | "
      f"warmup: {warmup_steps_b:,}")


@torch.no_grad()
def evaluate_blstm(model, loader, device, label_pad=-100):
    model.eval()
    tp = fp = fn = tn = 0
    total_loss, n = 0.0, 0
    for batch in loader:
        word_ids = batch["word_ids"].to(device, non_blocking=True)
        char_ids = batch["char_ids"].to(device, non_blocking=True)
        lengths  = batch["lengths"].to(device, non_blocking=True)
        labels   = batch["labels"].to(device, non_blocking=True)

        out = model(word_ids, char_ids, lengths, labels=labels)
        total_loss += out["loss"].item(); n += 1
        out2 = model(word_ids, char_ids, lengths, labels=None)
        preds = out2["predictions"]

        labels_cpu = labels.cpu().tolist()
        lengths_cpu = lengths.cpu().tolist()
        for i, seq_pred in enumerate(preds):
            L = lengths_cpu[i]
            for t in range(L):
                g = labels_cpu[i][t]
                if g == label_pad: continue
                p = seq_pred[t] if t < len(seq_pred) else 0
                if   g == 1 and p == 1: tp += 1
                elif g == 0 and p == 1: fp += 1
                elif g == 1 and p == 0: fn += 1
                else:                    tn += 1
    P = tp/(tp+fp) if (tp+fp) else 0.0
    R = tp/(tp+fn) if (tp+fn) else 0.0
    F1 = 2*P*R/(P+R) if (P+R) else 0.0
    acc = (tp+tn)/max(1, tp+fp+fn+tn)
    return {"loss": total_loss/max(1,n), "precision": P, "recall": R,
            "f1": F1, "accuracy": acc,
            "tp": tp, "fp": fp, "fn": fn, "tn": tn}


print("\n" + "=" * 60)
print("TRAINING BiLSTM-CRF")
print("=" * 60)

best_blstm_f1 = -1.0
blstm_history = []
t_total = time.time()

for epoch in range(1, BLSTM_EPOCHS + 1):
    blstm_model.train()
    t_epoch = time.time()
    run_loss, run_n = 0.0, 0
    log_loss, log_n = 0.0, 0
    LOG_EVERY_B = 200

    for step, batch in enumerate(train_blstm_loader, start=1):
        word_ids = batch["word_ids"].to(DEVICE, non_blocking=True)
        char_ids = batch["char_ids"].to(DEVICE, non_blocking=True)
        lengths  = batch["lengths"].to(DEVICE, non_blocking=True)
        labels   = batch["labels"].to(DEVICE, non_blocking=True)

        optimizer_b.zero_grad(set_to_none=True)
        out = blstm_model(word_ids, char_ids, lengths, labels=labels)
        loss = out["loss"]
        loss.backward()
        torch.nn.utils.clip_grad_norm_(blstm_model.parameters(), BLSTM_GRAD_CLIP)
        optimizer_b.step()
        scheduler_b.step()

        v = loss.item()
        run_loss += v; run_n += 1
        log_loss += v; log_n += 1
        if step % LOG_EVERY_B == 0:
            print(f"  epoch {epoch}  step {step:>4}/{len(train_blstm_loader)}  "
                  f"loss={log_loss/log_n:7.3f}  "
                  f"lr={optimizer_b.param_groups[0]['lr']:.2e}  "
                  f"elapsed={format_seconds(time.time()-t_epoch)}")
            log_loss, log_n = 0.0, 0

    train_loss = run_loss/max(1, run_n)
    epoch_time = time.time() - t_epoch
    val = evaluate_blstm(blstm_model, val_blstm_loader, DEVICE)

    blstm_history.append({"epoch": epoch, "train_loss": train_loss,
                          "val_loss": val["loss"], "val_p": val["precision"],
                          "val_r": val["recall"], "val_f1": val["f1"],
                          "val_acc": val["accuracy"],
                          "epoch_time": epoch_time})

    print(f"\n[BiLSTM Epoch {epoch}/{BLSTM_EPOCHS}]  "
          f"train_loss={train_loss:.4f}  val_loss={val['loss']:.4f}  "
          f"val_P={val['precision']:.4f}  val_R={val['recall']:.4f}  "
          f"val_F1={val['f1']:.4f}  val_acc={val['accuracy']:.4f}  "
          f"epoch_time={format_seconds(epoch_time)}")

    if val["f1"] > best_blstm_f1:
        best_blstm_f1 = val["f1"]
        torch.save({"epoch": epoch,
                    "model_state": blstm_model.state_dict(),
                    "best_val_f1": best_blstm_f1,
                    "val_metrics": val,
                    "word2id": word2id, "char2id": char2id,
                    "config": {"word_emb": WORD_EMB_DIM,
                               "char_emb": CHAR_EMB_DIM,
                               "char_filters": CHAR_CNN_FILTERS,
                               "char_widths": list(CHAR_CNN_WIDTHS),
                               "lstm_hidden": LSTM_HIDDEN,
                               "lstm_layers": LSTM_LAYERS,
                               "dropout": LSTM_DROPOUT,
                               "max_word_len": MAX_WORD_LEN,
                               "min_word_freq": MIN_WORD_FREQ}},
                   BLSTM_CKPT_PATH)
        print(f"           ✓ new best — saved to {BLSTM_CKPT_PATH}")
    print()

print("=" * 60)
print(f"BiLSTM-CRF training complete — total: "
      f"{format_seconds(time.time()-t_total)}")
print(f"Best val F1: {best_blstm_f1:.4f}")
print("=" * 60)


# ============================================================
# 7.6 Load best, evaluate on all test sets
# ============================================================
print("\nLoading best BiLSTM checkpoint and evaluating on test sets...")
b_ckpt = torch.load(BLSTM_CKPT_PATH, map_location=DEVICE, weights_only=False)
blstm_model.load_state_dict(b_ckpt["model_state"])
blstm_model.eval()

blstm_test_results = {}
for k in ["test1", "test2", "test3", "test4", "test5"]:
    res = evaluate_blstm(blstm_model, test_blstm_loaders[k], DEVICE)
    blstm_test_results[k] = res
    print(f"[{k}]  P={res['precision']:.4f}  R={res['recall']:.4f}  "
          f"F1={res['f1']:.4f}  acc={res['accuracy']:.4f}  "
          f"tp={res['tp']} fp={res['fp']} fn={res['fn']} tn={res['tn']}")


# ============================================================
# 7.7 Head-to-head comparison: JT Segmenter (raw) vs BiLSTM-CRF
# ============================================================
print("\n" + "=" * 60)
print("HEAD-TO-HEAD: JT Segmenter (XLM-R + CRF) vs BiLSTM-CRF baseline")
print("=" * 60)
print(f"{'Split':<7} | "
      f"{'JT_P':>6}  {'JT_R':>6}  {'JT_F1':>6}  ||  "
      f"{'BL_P':>6}  {'BL_R':>6}  {'BL_F1':>6}  ||  "
      f"{'ΔF1':>7}")
print("-" * 80)
for k in ["test1", "test2", "test3", "test4", "test5"]:
    jt = test_raw_results[k]
    bl = blstm_test_results[k]
    delta = jt["F1"] - bl["f1"]
    print(f"{k:<7} | "
          f"{jt['P']:>6.4f}  {jt['R']:>6.4f}  {jt['F1']:>6.4f}  ||  "
          f"{bl['precision']:>6.4f}  {bl['recall']:>6.4f}  {bl['f1']:>6.4f}  ||  "
          f"{delta:>+7.4f}")

# Macro
jt_macro_f1 = sum(test_raw_results[k]["F1"] for k in test_raw_results)/len(test_raw_results)
bl_macro_f1 = sum(blstm_test_results[k]["f1"] for k in blstm_test_results)/len(blstm_test_results)
print("-" * 80)
print(f"{'MACRO':<7} |  "
      f"{'':>6}  {'':>6}  {jt_macro_f1:>6.4f}  ||  "
      f"{'':>6}  {'':>6}  {bl_macro_f1:>6.4f}  ||  "
      f"{jt_macro_f1 - bl_macro_f1:>+7.4f}")

print("\n✅ Cell 7 complete — BiLSTM-CRF baseline trained and compared. "
      "Ready for Cell 8 (production segmenter for mizo_unseg1.txt).")

BUILDING VOCABULARIES FROM TRAINING DATA
Word vocab size : 51,543  (coverage: 51,541/105,194 types)
Char vocab size : 141
Most common chars : ['a', 'h', 'n', 'i', 't', 'u', 'l', 'e', 'm', 'k', 'g', 'w', 'r', 'c', 'o', ',', 'p', '.', 's', 'd']

Encoding splits for BiLSTM...
train: 46,909 | val: 500 | test sets: [100, 100, 100, 100, 100]
train batches: 1,466 | val batches: 8

BUILDING BiLSTM-CRF
Total params     : 11,358,798
Trainable params : 11,358,798
  (compare: JT Segmenter = 278,045,194 — BiLSTM is 4.1% the size)
Epochs: 10 | total_steps: 14,660 | warmup: 733

TRAINING BiLSTM-CRF
  epoch 1  step  200/1466  loss= 18.208  lr=2.73e-04  elapsed=2m01s
  epoch 1  step  400/1466  loss=  2.305  lr=5.46e-04  elapsed=4m00s
  epoch 1  step  600/1466  loss=  1.277  lr=8.19e-04  elapsed=6m13s
  epoch 1  step  800/1466  loss=  0.816  lr=9.95e-04  elapsed=8m18s
  epoch 1  step 1000/1466  loss=  0.657  lr=9.81e-04  elapsed=10m35s
  epoch 1  step 1200/1466  loss=  0.522  lr=9.66e-04  elapsed=12m39s

### Cell 8: Production segmenter for mizo_unseg1.txt

In [9]:
# ============================================================
# Cell 8: Production segmenter for mizo_unseg1.txt
# Self-contained: defines its own helpers + reuses model in memory.
# ============================================================

# ---------- Self-contained helpers (survive kernel restart) ----------
def format_seconds(s):
    m, s = divmod(int(s), 60)
    h, m = divmod(m, 60)
    return f"{h:d}h{m:02d}m{s:02d}s" if h else f"{m:d}m{s:02d}s"


# ---------- Configuration ----------
INPUT_FILE       = Path("data") / "training_unseg.txt"   # adjust if elsewhere
OUTPUT_DIR       = Path("output_segmented")
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_TXT_PATH  = OUTPUT_DIR / "mizo_seg1_jt.txt"      # one sentence per line
OUTPUT_JSONL_PATH = OUTPUT_DIR / "mizo_seg1_jt.jsonl"   # paragraph-aligned audit trail
INFER_BATCH      = 32
MAX_PARA_TOKENS  = 1000   # safety cap; longer paragraphs get split-warned

assert INPUT_FILE.exists(), (
    f"Input file not found: {INPUT_FILE.resolve()}\n"
    f"Adjust INPUT_FILE at the top of this cell."
)


# ============================================================
# 8.1 Read raw paragraphs
# ============================================================
print("=" * 60)
print(f"READING INPUT: {INPUT_FILE}")
print("=" * 60)

def read_raw_paragraphs(path):
    """
    Read mizo_unseg1.txt. Treats each non-empty line as one paragraph,
    stripping <p>...</p> tags if present (matching the training format),
    but also tolerates files without tags.
    """
    paragraphs = []
    with open(path, "r", encoding="utf-8") as f:
        for ln in f:
            s = ln.rstrip("\n").strip()
            if not s:
                continue
            if s.startswith("<p>"): s = s[3:]
            if s.endswith("</p>"):  s = s[:-4]
            s = s.strip()
            if s:
                paragraphs.append(s)
    return paragraphs

raw_paragraphs = read_raw_paragraphs(INPUT_FILE)
print(f"Paragraphs read : {len(raw_paragraphs):,}")
ws_token_counts = [len(p.split()) for p in raw_paragraphs]
print(f"Token stats     : min={min(ws_token_counts):,}  "
      f"max={max(ws_token_counts):,}  "
      f"mean={np.mean(ws_token_counts):.1f}  "
      f"median={int(np.median(ws_token_counts))}  "
      f"total={sum(ws_token_counts):,}")
n_long = sum(1 for n in ws_token_counts if n > MAX_PARA_TOKENS)
if n_long:
    print(f"WARN: {n_long} paragraphs exceed {MAX_PARA_TOKENS} tokens — "
          f"will still be segmented, but consider pre-splitting if results look poor.")


# ============================================================
# 8.2 Encode paragraphs for inference (no labels — uses dummy 0s)
# ============================================================
print("\n" + "=" * 60)
print("ENCODING FOR INFERENCE")
print("=" * 60)

infer_records = []   # one per paragraph: {"pid","tokens","chunks":[...]}
for pid, para in enumerate(raw_paragraphs):
    tokens = para.split()
    if not tokens:
        continue
    # Dummy labels (0s) — only used to satisfy encode_paragraph's signature;
    # we discard predictions and just need the chunk structure.
    dummy_labels = [0] * len(tokens)
    chunks = encode_paragraph(tokens, dummy_labels, tokenizer,
                              max_len=MAX_LEN, stride=STRIDE,
                              label_pad=LABEL_PAD)
    for ch in chunks:
        ch["pid"]     = pid
        ch["n_words"] = len(tokens)
    infer_records.append({"pid": pid, "tokens": tokens, "chunks": chunks})

flat_chunks = [ch for r in infer_records for ch in r["chunks"]]
print(f"Total chunks    : {len(flat_chunks):,}")
print(f"Long paragraphs : "
      f"{sum(1 for r in infer_records if len(r['chunks']) > 1):,}")

infer_ds     = MizoSegDataset(flat_chunks)
infer_loader = DataLoader(infer_ds, batch_size=INFER_BATCH, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                          collate_fn=collate_fn)
print(f"Inference batches: {len(infer_loader):,} "
      f"(batch_size={INFER_BATCH})")


# ============================================================
# 8.3 Run inference + reassemble word-level predictions
# ============================================================
print("\n" + "=" * 60)
print("RUNNING JT SEGMENTER")
print("=" * 60)

model.eval()
t0 = time.time()
preds_by_pid = predict_paragraphs(model, infer_loader, DEVICE)
infer_time = time.time() - t0
print(f"Inference time  : {format_seconds(infer_time)}  "
      f"({len(raw_paragraphs) / max(infer_time, 1e-6):.1f} paragraphs/sec)")


# ============================================================
# 8.4 Split paragraphs into sentences using predicted boundaries
# ============================================================
def split_by_predictions(tokens, preds):
    """
    Walk through tokens; whenever preds[i]==1, end the current sentence
    inclusive of token i, then start a new one. Trailing tokens after the
    last 1 (if any — should not happen post-reassembly) are appended to the
    final sentence.
    """
    sentences = []
    cur = []
    for tok, p in zip(tokens, preds):
        cur.append(tok)
        if p == 1:
            sentences.append(" ".join(cur))
            cur = []
    if cur:
        sentences.append(" ".join(cur))
    return sentences

print("\n" + "=" * 60)
print("WRITING OUTPUTS")
print("=" * 60)

n_sent_total = 0
n_para_with_no_boundary = 0

with open(OUTPUT_TXT_PATH, "w", encoding="utf-8") as f_txt, \
     open(OUTPUT_JSONL_PATH, "w", encoding="utf-8") as f_jsonl:
    for pid, para in enumerate(raw_paragraphs):
        tokens = para.split()
        preds  = preds_by_pid.get(pid, [0] * len(tokens))
        # Safety: ensure last token is a boundary (paragraph ends a sentence)
        if tokens and preds[-1] == 0:
            preds = preds[:]            # don't mutate the cached prediction
            preds[-1] = 1
        if sum(preds) == 0:
            n_para_with_no_boundary += 1
            preds[-1] = 1               # force one boundary

        sentences = split_by_predictions(tokens, preds)
        n_sent_total += len(sentences)

        # Write plain text — one sentence per line, blank line between paragraphs
        for s in sentences:
            f_txt.write(s + "\n")
        f_txt.write("\n")               # paragraph separator

        # Write JSONL — paragraph-aligned audit trail
        f_jsonl.write(json.dumps({
            "pid":       pid,
            "n_tokens":  len(tokens),
            "n_sentences": len(sentences),
            "sentences": sentences,
        }, ensure_ascii=False) + "\n")

print(f"Paragraphs in   : {len(raw_paragraphs):,}")
print(f"Sentences out   : {n_sent_total:,}  "
      f"(avg {n_sent_total / max(1, len(raw_paragraphs)):.2f} sent/para)")
if n_para_with_no_boundary:
    print(f"WARN: {n_para_with_no_boundary} paragraphs had zero predicted "
          f"boundaries → forced final-token boundary.")
print(f"Wrote plain text: {OUTPUT_TXT_PATH}")
print(f"Wrote JSONL     : {OUTPUT_JSONL_PATH}")


# ============================================================
# 8.5 Sanity peek — first 3 paragraphs, before/after
# ============================================================
print("\n" + "=" * 60)
print("SANITY PEEK — first 3 paragraphs")
print("=" * 60)
for pid in range(min(3, len(raw_paragraphs))):
    para   = raw_paragraphs[pid]
    tokens = para.split()
    preds  = preds_by_pid.get(pid, [0] * len(tokens))
    if tokens and preds[-1] == 0:
        preds = preds[:]; preds[-1] = 1
    sents = split_by_predictions(tokens, preds)
    print(f"\n--- Paragraph {pid}  ({len(tokens)} tokens → {len(sents)} sentences) ---")
    print(f"INPUT : {para[:200]}{'...' if len(para) > 200 else ''}")
    print("SPLIT :")
    for i, s in enumerate(sents, 1):
        preview = s if len(s) <= 200 else s[:200] + "..."
        print(f"  [{i}] {preview}")


# ============================================================
# 8.6 Write a tiny results-snapshot JSON for the thesis appendix
# ============================================================
RESULTS_JSON = OUTPUT_DIR / "jt_segmenter_results.json"
results_snapshot = {
    "model": {
        "name":       "JT Segmenter",
        "encoder":    MODEL_NAME,
        "head":       "Linear(768→2) + Linear-Chain CRF",
        "num_labels": NUM_LABELS,
        "max_len":    MAX_LEN,
        "stride":     STRIDE,
        "dropout":    DROPOUT,
        "total_params":     278_045_194,
        "trainable_params": 278_045_194,
        "best_epoch":        ckpt["epoch"],
        "best_val_f1":       float(ckpt["best_val_f1"]),
    },
    "training": {
        "data":          "Mizo paragraph-aligned corpus (this work)",
        "train_paragraphs":    len(train_data),
        "val_paragraphs":      len(val_data),
        "test_paragraphs":     {k: len(v) for k, v in test_data.items()},
        "boundary_class_ratio": "~5.91% (~1:16)",
        "epochs":         5,
        "batch_size_train": 24,
        "lr_encoder":     2e-5,
        "lr_head":        5e-4,
        "weight_decay":   0.01,
        "warmup_ratio":   0.1,
        "grad_clip":      1.0,
        "amp":            True,
    },
    "test_results_jt_segmenter": {
        k: {"P": float(v["P"]), "R": float(v["R"]), "F1": float(v["F1"]),
            "paragraph_exact": f"{v['para_exact']}/{v['para_total']}"}
        for k, v in test_raw_results.items()
    },
    "test_results_bilstm_crf_baseline": {
        k: {"P": float(v["precision"]), "R": float(v["recall"]),
            "F1": float(v["f1"])}
        for k, v in blstm_test_results.items()
    },
    "head_to_head_macro_f1": {
        "jt_segmenter":    float(jt_macro_f1),
        "bilstm_crf":      float(bl_macro_f1),
        "delta":           float(jt_macro_f1 - bl_macro_f1),
    },
    "post_processor_finding": {
        "rule_proposed": "Demote 'a,'/'va,' tokens from boundary candidates",
        "rule_outcome":  "Refuted empirically",
        "evidence_train_pct": "14.46% of gold boundaries terminate in 'a,'/'va,'",
        "evidence_test_pct_range": "13.22%–18.20%",
        "decision":      "Post-processor removed; raw model output is final.",
    },
    "deployment_run": {
        "input_file":      str(INPUT_FILE),
        "input_paragraphs": len(raw_paragraphs),
        "output_sentences": n_sent_total,
        "output_txt":      str(OUTPUT_TXT_PATH),
        "output_jsonl":    str(OUTPUT_JSONL_PATH),
        "inference_seconds": round(infer_time, 2),
    },
}
with open(RESULTS_JSON, "w", encoding="utf-8") as f:
    json.dump(results_snapshot, f, indent=2, ensure_ascii=False)
print(f"\nWrote results snapshot: {RESULTS_JSON}")


# ============================================================
# 8.7 Reusable function (for future use elsewhere)
# ============================================================
def segment_text(text, model=model, tokenizer=tokenizer, device=DEVICE):
    """
    Segment a single paragraph (or multi-paragraph string split on newlines)
    of raw Mizo text into sentences using JT Segmenter.

    Returns: list[list[str]] — outer list is paragraphs, inner list is sentences.
    """
    paragraphs = [ln.strip() for ln in text.split("\n") if ln.strip()]
    if not paragraphs:
        return []

    records = []
    for pid, para in enumerate(paragraphs):
        toks = para.split()
        if not toks: continue
        chunks = encode_paragraph(toks, [0]*len(toks), tokenizer,
                                  max_len=MAX_LEN, stride=STRIDE,
                                  label_pad=LABEL_PAD)
        for ch in chunks:
            ch["pid"] = pid; ch["n_words"] = len(toks)
        records.append({"pid": pid, "tokens": toks, "chunks": chunks})

    flat = [ch for r in records for ch in r["chunks"]]
    loader = DataLoader(MizoSegDataset(flat), batch_size=INFER_BATCH,
                        shuffle=False, collate_fn=collate_fn)
    pred_map = predict_paragraphs(model, loader, device)

    out = []
    for r in records:
        toks  = r["tokens"]
        preds = pred_map.get(r["pid"], [0]*len(toks))
        if toks and preds[-1] == 0:
            preds = preds[:]; preds[-1] = 1
        if sum(preds) == 0:
            preds[-1] = 1
        out.append(split_by_predictions(toks, preds))
    return out


# Demo of the reusable function
print("\n" + "=" * 60)
print("REUSABLE FUNCTION DEMO — segment_text()")
print("=" * 60)
demo_text = (
    "Pathian chu kan Pa a ni a, ani chuan kan ṭanpui zel ang. "
    "He hun a ngaihnawm hle a, kan i pawmpui tlat tur a ni."
)
demo_out = segment_text(demo_text)
print(f"Input : {demo_text}")
for pi, sents in enumerate(demo_out):
    for si, s in enumerate(sents, 1):
        print(f"  Para {pi} Sent {si}: {s}")

print("\n✅ Cell 8 complete — JT Segmenter deployed on mizo_unseg1.txt. "
      "Outputs in 'output_segmented/' and reusable segment_text() in memory.")

READING INPUT: data\training_unseg.txt
Paragraphs read : 46,909
Token stats     : min=2  max=3,248  mean=65.4  median=27  total=3,069,899
WARN: 31 paragraphs exceed 1000 tokens — will still be segmented, but consider pre-splitting if results look poor.

ENCODING FOR INFERENCE
Total chunks    : 57,218
Long paragraphs : 6,829
Inference batches: 1,789 (batch_size=32)

RUNNING JT SEGMENTER
Inference time  : 11m17s  (69.2 paragraphs/sec)

WRITING OUTPUTS
Paragraphs in   : 46,909
Sentences out   : 182,210  (avg 3.88 sent/para)
Wrote plain text: output_segmented\mizo_seg1_jt.txt
Wrote JSONL     : output_segmented\mizo_seg1_jt.jsonl

SANITY PEEK — first 3 paragraphs

--- Paragraph 0  (168 tokens → 9 sentences) ---
INPUT : He lehkhabu hi chu Manipur-a an awm tawh loh hnua a ziak a ni a. Manipura an awm lai, 1997 velah Kuki leh Paihte unaute nasa takin an inbei a, mi tam takin nunna an chân phah a. Hemi a ngaihtuahna kal...
SPLIT :
  [1] He lehkhabu hi chu Manipur-a an awm tawh loh hnua a ziak a